# 3D Gaussian Data Converter

This script converts a parquet data from [the unofficial implementation **taichi_3d_gaussian_splatting**](https://github.com/wanmeihuali/taichi_3d_gaussian_splatting) into the ply format of [the official implementation **3D Gaussian Splatting**](https://github.com/graphdeco-inria/gaussian-splatting)

In [26]:
import os
import numpy as np
from plyfile import PlyData, PlyElement
import argparse
import struct
import math

def convert_gsplat_to_3dgs(input_path, output_path, options):
    """
    GSplatで生成されたPLYファイルをオリジナルの3DGSと互換性のある形式に変換
    
    Args:
        input_path: 入力PLYファイルのパス
        output_path: 出力PLYファイルのパス
        options: 変換オプション
    """
    print(f"変換開始: {input_path} -> {output_path}")
    
    # PLYファイルを読み込み
    try:
        ply_data = PlyData.read(input_path)
    except Exception as e:
        print(f"エラー: PLYファイルの読み込みに失敗しました - {e}")
        return False
    
    vertices = ply_data['vertex']
    vertex_count = len(vertices)
    print(f"頂点数: {vertex_count}")
    
    # プロパティ情報を収集
    old_props = [prop.name for prop in vertices.properties]
    print(f"入力ファイルのプロパティ: {old_props}")
    
    # プロパティをグループ化
    pos_props = [p for p in old_props if p in ['x', 'y', 'z']]
    normal_props = [p for p in old_props if p in ['nx', 'ny', 'nz']]
    
    # GSplatの命名規則を検出（fdcX または f_dc_X）
    dc_pattern = 'fdc' if any(p.startswith('fdc') for p in old_props) else 'f_dc_'
    rest_pattern = 'f*rest*' if any(p.startswith('f*rest*') for p in old_props) else 'f_rest_'
    scale_pattern = 'scale*' if any(p.startswith('scale*') for p in old_props) else 'scale_'
    rot_pattern = 'rot*' if any(p.startswith('rot*') for p in old_props) else 'rot_'
    
    dc_props = [p for p in old_props if p.startswith(dc_pattern)]
    rest_props = [p for p in old_props if p.startswith(rest_pattern)]
    scale_props = [p for p in old_props if p.startswith(scale_pattern)]
    rot_props = [p for p in old_props if p.startswith(rot_pattern)]
    opacity_prop = 'opacity' if 'opacity' in old_props else None
    
    print(f"DC成分: {len(dc_props)}, REST成分: {len(rest_props)}")
    print(f"スケール: {len(scale_props)}, 回転: {len(rot_props)}")
    
    # プロパティ名のマッピング
    prop_map = {}
    for prop in old_props:
        if prop.startswith('fdc'):
            prop_map[prop] = f"f_dc_{prop[3:]}"
        elif prop.startswith('f*rest*'):
            prop_map[prop] = f"f_rest_{prop[7:]}"
        elif prop.startswith('scale*'):
            prop_map[prop] = f"scale_{prop[6:]}"
        elif prop.startswith('rot*'):
            prop_map[prop] = f"rot_{prop[4:]}"
        else:
            prop_map[prop] = prop
    
    # データの準備
    position = np.zeros((vertex_count, 3))
    for i, prop in enumerate(['x', 'y', 'z']):
        if prop in old_props:
            position[:, i] = vertices[prop]
    
    normal = np.zeros((vertex_count, 3))
    for i, prop in enumerate(['nx', 'ny', 'nz']):
        if prop in old_props:
            normal[:, i] = vertices[prop]
    
    # 座標変換（必要な場合）
    if options.get('swap_axes', False):
        mode = options.get('swap_mode', 'y_up_to_z_up')
        if mode == 'y_up_to_z_up':
            # Y-upからZ-upへの変換（Y→Z, Z→-Y）
            position[:, [1, 2]] = position[:, [2, 1]]
            position[:, 2] *= -1
            normal[:, [1, 2]] = normal[:, [2, 1]]
            normal[:, 2] *= -1
        elif mode == 'z_up_to_y_up':
            # Z-upからY-upへの変換（Z→Y, Y→-Z）
            position[:, [1, 2]] = position[:, [2, 1]]
            position[:, 1] *= -1
            normal[:, [1, 2]] = normal[:, [2, 1]]
            normal[:, 1] *= -1
    
    # 色彩データの抽出と変換
    if dc_props:
        # DCプロパティをソート
        dc_props = sorted(dc_props, key=lambda p: int(p.replace(dc_pattern, '')) if p.replace(dc_pattern, '').isdigit() else 0)
        dc_values = np.zeros((vertex_count, len(dc_props)))
        
        for i, prop in enumerate(dc_props):
            dc_values[:, i] = vertices[prop]
        
        # カラースペース変換（必要な場合）
        if options.get('color_space_conversion', False):
            mode = options.get('color_mode', 'linear_to_srgb')
            if mode == 'linear_to_srgb':
                # 線形RGBからsRGBへの変換
                mask = dc_values > 0.0031308
                dc_values[mask] = 1.055 * np.power(dc_values[mask], 1.0/2.4) - 0.055
                dc_values[~mask] = dc_values[~mask] * 12.92
            elif mode == 'srgb_to_linear':
                # sRGBから線形RGBへの変換
                mask = dc_values > 0.04045
                dc_values[mask] = np.power((dc_values[mask] + 0.055) / 1.055, 2.4)
                dc_values[~mask] = dc_values[~mask] / 12.92
    
    # 球面調和関数係数の抽出と変換
    if rest_props:
        # RESTプロパティをソート
        rest_props = sorted(rest_props, key=lambda p: int(p.replace(rest_pattern, '')) if p.replace(rest_pattern, '').isdigit() else 0)
        rest_values = np.zeros((vertex_count, len(rest_props)))
        
        for i, prop in enumerate(rest_props):
            rest_values[:, i] = vertices[prop]
        
        # SH係数の変換（最も重要な部分）
        rest_per_channel = len(rest_props) // 3
        sh_degree = int(np.sqrt(rest_per_channel + 1)) - 1
        print(f"推定SH次数: {sh_degree}, 各チャンネルの係数数: {rest_per_channel}")
        
        # SH係数の変換方法の選択
        sh_mode = options.get('sh_conversion', 'gsplat_to_3dgs')
        rest_values = convert_sh_coefficients(rest_values, sh_degree, sh_mode)
    
    # 不透明度の抽出と変換
    opacity = None
    if opacity_prop:
        opacity = vertices[opacity_prop]
        # 不透明度の調整（必要な場合）
        if options.get('opacity_scale', 1.0) != 1.0:
            opacity *= options.get('opacity_scale')
    
    # スケールの抽出と変換
    scale = None
    if scale_props:
        # スケールプロパティをソート
        scale_props = sorted(scale_props, key=lambda p: int(p.replace(scale_pattern, '')) if p.replace(scale_pattern, '').isdigit() else 0)
        scale = np.zeros((vertex_count, len(scale_props)))
        
        for i, prop in enumerate(scale_props):
            scale[:, i] = vertices[prop]
        
        # スケールの調整（必要な場合）
        if options.get('scale_factor', 1.0) != 1.0:
            scale *= options.get('scale_factor')
    
    # 回転の抽出と変換
    rotation = None
    if rot_props:
        # 回転プロパティをソート
        rot_props = sorted(rot_props, key=lambda p: int(p.replace(rot_pattern, '')) if p.replace(rot_pattern, '').isdigit() else 0)
        rotation = np.zeros((vertex_count, len(rot_props)))
        
        for i, prop in enumerate(rot_props):
            rotation[:, i] = vertices[prop]
        
        # クォータニオンの規約変換
        if options.get('quaternion_convention', False):
            # GSplatとオリジナル3DGSのクォータニオン規約が異なる場合
            # 例: wxyz <-> xyzw 形式の変換
            if len(rot_props) == 4:  # クォータニオン形式の場合
                rotation = convert_quaternion(rotation, options.get('quat_mode', 'wxyz_to_xyzw'))
        
        # クォータニオンの正規化
        if options.get('normalize_quaternion', False) and len(rot_props) == 4:
            quat_norm = np.sqrt(np.sum(rotation**2, axis=1, keepdims=True))
            quat_norm = np.where(quat_norm > 0, quat_norm, 1.0)
            rotation /= quat_norm
    
    # 新しいデータの構築
    data_dict = {}
    
    # 位置データ
    for i, prop in enumerate(['x', 'y', 'z']):
        data_dict[prop] = position[:, i]
    
    # 法線データ
    for i, prop in enumerate(['nx', 'ny', 'nz']):
        data_dict[prop] = normal[:, i]
    
    # 色彩データ (DC成分)
    if dc_props:
        for i, prop in enumerate(dc_props):
            data_dict[prop_map[prop]] = dc_values[:, i]
    
    # SH係数 (REST成分)
    if rest_props:
        for i, prop in enumerate(rest_props):
            data_dict[prop_map[prop]] = rest_values[:, i]
    
    # 不透明度
    if opacity_prop:
        data_dict[opacity_prop] = opacity
    
    # スケール
    if scale_props:
        for i, prop in enumerate(scale_props):
            data_dict[prop_map[prop]] = scale[:, i]
    
    # 回転
    if rot_props:
        for i, prop in enumerate(rot_props):
            data_dict[prop_map[prop]] = rotation[:, i]
    
    # 新しいプロパティ名の並び順を決定
    # 3DGSの標準的な順序に合わせる
    new_props = []
    for p in ['x', 'y', 'z']:
        new_props.append(p)
    for p in ['nx', 'ny', 'nz']:
        new_props.append(p)
    for p in sorted(dc_props, key=lambda p: int(p.replace(dc_pattern, '')) if p.replace(dc_pattern, '').isdigit() else 0):
        new_props.append(prop_map[p])
    for p in sorted(rest_props, key=lambda p: int(p.replace(rest_pattern, '')) if p.replace(rest_pattern, '').isdigit() else 0):
        new_props.append(prop_map[p])
    if opacity_prop:
        new_props.append(opacity_prop)
    for p in sorted(scale_props, key=lambda p: int(p.replace(scale_pattern, '')) if p.replace(scale_pattern, '').isdigit() else 0):
        new_props.append(prop_map[p])
    for p in sorted(rot_props, key=lambda p: int(p.replace(rot_pattern, '')) if p.replace(rot_pattern, '').isdigit() else 0):
        new_props.append(prop_map[p])
    
    print(f"出力プロパティ: {new_props}")
    
    # 新しいデータ型の定義
    dtype_full = [(name, 'f4') for name in new_props]
    
    # PLYファイル用のデータ配列の作成
    elements = np.empty(vertex_count, dtype=dtype_full)
    
    # データ配列にデータを格納
    for name in new_props:
        elements[name] = data_dict[name]
    
    # PLYファイルとして保存
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(output_path)
    
    print(f"変換完了: {output_path}")
    return True

def convert_sh_coefficients(values, sh_degree, mode='gsplat_to_3dgs'):
    """
    球面調和関数係数の変換
    
    Args:
        values: 元の係数値の配列 (N, K)
        sh_degree: SH次数
        mode: 変換モード
    
    Returns:
        変換後の係数値の配列 (N, K)
    """
    result = values.copy()
    rest_per_channel = (sh_degree + 1) ** 2 - 1  # DC成分を除く
    num_channels = values.shape[1] // rest_per_channel
    
    if mode == 'gsplat_to_3dgs':
        # GSplatから3DGSへの変換
        # GSplatは球面調和関数の係数をR->G->B方向で格納し、m値に関する規約が異なる
        
        for ch in range(num_channels):
            ch_offset = ch * rest_per_channel
            
            # 各次数ごとに係数を処理
            for l in range(1, sh_degree + 1):
                l_offset = l * l - 1  # 次数lの係数の開始インデックス
                l_size = 2 * l + 1    # 次数lの係数の数
                
                # GSplatとオリジナル3DGSでは球面調和関数のm値の順序が異なる
                # m値が反転している可能性がある（GSplatからオリジナル3DGSへ）
                for m_idx in range(l_size):
                    m = m_idx - l  # -l, -l+1, ..., 0, 1, ..., l
                    
                    # 反転したm値
                    m_flipped = -m
                    
                    # オリジナルのインデックス
                    orig_idx = ch_offset + l_offset + m_idx
                    
                    # 反転後のインデックス（m_flippedが-l〜lの範囲内）
                    flipped_m_idx = m_flipped + l  # 0〜2lのインデックスに変換
                    flipped_idx = ch_offset + l_offset + flipped_m_idx
                    
                    # インデックスが有効範囲内の場合のみ適用
                    if 0 <= flipped_idx < values.shape[1]:
                        result[:, orig_idx] = values[:, flipped_idx]
    
    elif mode == '3dgs_to_gsplat':
        # 3DGSからGSplatへの変換（上記の逆操作）
        for ch in range(num_channels):
            ch_offset = ch * rest_per_channel
            
            for l in range(1, sh_degree + 1):
                l_offset = l * l - 1
                l_size = 2 * l + 1
                
                for m_idx in range(l_size):
                    m = m_idx - l
                    m_flipped = -m
                    
                    orig_idx = ch_offset + l_offset + m_idx
                    flipped_m_idx = m_flipped + l
                    flipped_idx = ch_offset + l_offset + flipped_m_idx
                    
                    if 0 <= flipped_idx < values.shape[1]:
                        result[:, flipped_idx] = values[:, orig_idx]
    
    elif mode == 'reorder_channels':
        # RGB各チャンネル内で係数の並び順を変更
        # 例：チャンネル内で係数を反転
        for ch in range(num_channels):
            ch_start = ch * rest_per_channel
            ch_end = (ch + 1) * rest_per_channel
            
            # チャンネル内で係数を反転
            for i in range(rest_per_channel):
                result[:, ch_start + i] = values[:, ch_end - 1 - i]
    
    elif mode == 'swap_channels':
        # RGBチャンネル間の並び替え（例：RGB -> BGR）
        for i in range(values.shape[0]):
            for ch in range(num_channels):
                ch_start = ch * rest_per_channel
                ch_end = (ch + 1) * rest_per_channel
                
                # 新しいチャンネル番号（例：0->2, 1->1, 2->0 でRGB->BGR）
                new_ch = num_channels - 1 - ch
                new_ch_start = new_ch * rest_per_channel
                
                result[i, new_ch_start:new_ch_start+rest_per_channel] = values[i, ch_start:ch_end]
    
    elif mode == 'normalize':
        # 係数の正規化
        for ch in range(num_channels):
            ch_offset = ch * rest_per_channel
            
            # 各次数ごとに係数をスケーリング
            for l in range(1, sh_degree + 1):
                l_offset = l * l - 1
                l_size = 2 * l + 1
                
                # 次数lの係数に対する正規化ファクター（例：SH基底関数の差異を補正）
                # 具体的な値はソフトウェアの実装に依存
                factor = 1.0 / (2*l + 1)
                
                for m_idx in range(l_size):
                    idx = ch_offset + l_offset + m_idx
                    result[:, idx] *= factor
    
    return result

def convert_quaternion(quats, mode='wxyz_to_xyzw'):
    """
    クォータニオンの規約変換
    
    Args:
        quats: クォータニオン値の配列 (N, 4)
        mode: 変換モード ('wxyz_to_xyzw' または 'xyzw_to_wxyz')
    
    Returns:
        変換後のクォータニオン値の配列 (N, 4)
    """
    result = quats.copy()
    
    if mode == 'wxyz_to_xyzw':
        # wxyz -> xyzw 形式に変換
        result = quats[:, [1, 2, 3, 0]]
    elif mode == 'xyzw_to_wxyz':
        # xyzw -> wxyz 形式に変換
        result = quats[:, [3, 0, 1, 2]]
    elif mode == 'conjugate':
        # クォータニオンの共役（虚部の符号を反転）
        result[:, 1:] *= -1
    
    return result

def try_all_conversions(input_path, output_dir):
    """
    様々な変換パターンを試す
    
    Args:
        input_path: 入力PLYファイルのパス
        output_dir: 出力ディレクトリ
    """
    os.makedirs(output_dir, exist_ok=True)
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    
    # 基本オプション
    base_options = {
        'swap_axes': False,
        'color_space_conversion': False,
        'quaternion_convention': False,
        'normalize_quaternion': False,
        'scale_factor': 1.0,
        'opacity_scale': 1.0,
        'sh_conversion': 'gsplat_to_3dgs'
    }
    
    # バリエーションの定義
    variants = []
    
    # バリエーション1: 基本変換（プロパティ名のみ）
    variants.append({
        'name': 'basic',
        'description': '基本変換（プロパティ名のみ）',
        'options': base_options.copy()
    })
    
    # バリエーション2: SH係数変換（m値の反転）
    variants.append({
        'name': 'sh_flip_m',
        'description': 'SH係数変換（m値の反転）',
        'options': {**base_options, 'sh_conversion': 'gsplat_to_3dgs'}
    })
    
    # バリエーション3: SH係数変換（チャンネル内で反転）
    variants.append({
        'name': 'sh_reorder_channels',
        'description': 'SH係数変換（チャンネル内で反転）',
        'options': {**base_options, 'sh_conversion': 'reorder_channels'}
    })
    
    # バリエーション4: SH係数変換（チャンネル間の入れ替え）
    variants.append({
        'name': 'sh_swap_channels',
        'description': 'SH係数変換（チャンネル間の入れ替え）',
        'options': {**base_options, 'sh_conversion': 'swap_channels'}
    })
    
    # バリエーション5: 座標系変換（Y-up -> Z-up）
    variants.append({
        'name': 'y_up_to_z_up',
        'description': '座標系変換（Y-up -> Z-up）',
        'options': {**base_options, 'swap_axes': True, 'swap_mode': 'y_up_to_z_up'}
    })
    
    # バリエーション6: 座標系変換（Z-up -> Y-up）
    variants.append({
        'name': 'z_up_to_y_up',
        'description': '座標系変換（Z-up -> Y-up）',
        'options': {**base_options, 'swap_axes': True, 'swap_mode': 'z_up_to_y_up'}
    })
    
    # バリエーション7: カラースペース変換（線形RGB -> sRGB）
    variants.append({
        'name': 'linear_to_srgb',
        'description': 'カラースペース変換（線形RGB -> sRGB）',
        'options': {**base_options, 'color_space_conversion': True, 'color_mode': 'linear_to_srgb'}
    })
    
    # バリエーション8: カラースペース変換（sRGB -> 線形RGB）
    variants.append({
        'name': 'srgb_to_linear',
        'description': 'カラースペース変換（sRGB -> 線形RGB）',
        'options': {**base_options, 'color_space_conversion': True, 'color_mode': 'srgb_to_linear'}
    })
    
    # バリエーション9: クォータニオン規約変換（wxyz -> xyzw）
    variants.append({
        'name': 'quat_wxyz_to_xyzw',
        'description': 'クォータニオン規約変換（wxyz -> xyzw）',
        'options': {**base_options, 'quaternion_convention': True, 'quat_mode': 'wxyz_to_xyzw'}
    })
    
    # バリエーション10: クォータニオン規約変換（xyzw -> wxyz）
    variants.append({
        'name': 'quat_xyzw_to_wxyz',
        'description': 'クォータニオン規約変換（xyzw -> wxyz）',
        'options': {**base_options, 'quaternion_convention': True, 'quat_mode': 'xyzw_to_wxyz'}
    })
    
    # バリエーション11: クォータニオン共役
    variants.append({
        'name': 'quat_conjugate',
        'description': 'クォータニオン共役',
        'options': {**base_options, 'quaternion_convention': True, 'quat_mode': 'conjugate'}
    })
    
    # バリエーション12: スケール調整（2倍）
    variants.append({
        'name': 'scale_2x',
        'description': 'スケール調整（2倍）',
        'options': {**base_options, 'scale_factor': 2.0}
    })
    
    # バリエーション13: スケール調整（0.5倍）
    variants.append({
        'name': 'scale_0.5x',
        'description': 'スケール調整（0.5倍）',
        'options': {**base_options, 'scale_factor': 0.5}
    })
    
    # バリエーション14: 不透明度調整（0.8倍）
    variants.append({
        'name': 'opacity_0.8x',
        'description': '不透明度調整（0.8倍）',
        'options': {**base_options, 'opacity_scale': 0.8}
    })
    
    # 組み合わせバリエーション
    
    # SH反転 + Y-up -> Z-up
    variants.append({
        'name': 'sh_flip_m_y_up_to_z_up',
        'description': 'SH係数反転 + 座標系変換（Y-up -> Z-up）',
        'options': {
            **base_options,
            'sh_conversion': 'gsplat_to_3dgs',
            'swap_axes': True,
            'swap_mode': 'y_up_to_z_up'
        }
    })
    
    # SH反転 + Z-up -> Y-up
    variants.append({
        'name': 'sh_flip_m_z_up_to_y_up',
        'description': 'SH係数反転 + 座標系変換（Z-up -> Y-up）',
        'options': {
            **base_options,
            'sh_conversion': 'gsplat_to_3dgs',
            'swap_axes': True,
            'swap_mode': 'z_up_to_y_up'
        }
    })
    
    # SH反転 + クォータニオン規約変換（wxyz -> xyzw）
    variants.append({
        'name': 'sh_flip_m_quat_wxyz_to_xyzw',
        'description': 'SH係数反転 + クォータニオン規約変換（wxyz -> xyzw）',
        'options': {
            **base_options,
            'sh_conversion': 'gsplat_to_3dgs',
            'quaternion_convention': True,
            'quat_mode': 'wxyz_to_xyzw'
        }
    })
    
    # SH反転 + クォータニオン規約変換（xyzw -> wxyz）
    variants.append({
        'name': 'sh_flip_m_quat_xyzw_to_wxyz',
        'description': 'SH係数反転 + クォータニオン規約変換（xyzw -> wxyz）',
        'options': {
            **base_options,
            'sh_conversion': 'gsplat_to_3dgs',
            'quaternion_convention': True,
            'quat_mode': 'xyzw_to_wxyz'
        }
    })
    
    # SH反転 + 線形RGB -> sRGB
    variants.append({
        'name': 'sh_flip_m_linear_to_srgb',
        'description': 'SH係数反転 + カラースペース変換（線形RGB -> sRGB）',
        'options': {
            **base_options,
            'sh_conversion': 'gsplat_to_3dgs',
            'color_space_conversion': True,
            'color_mode': 'linear_to_srgb'
        }
    })
    
    # 各バリエーションを適用
    for variant in variants:
        output_path = os.path.join(output_dir, f"{base_name}_{variant['name']}.ply")
        print(f"\nバリエーション: {variant['description']}")
        convert_gsplat_to_3dgs(input_path, output_path, variant['options'])
    
    print(f"\n合計 {len(variants)} バリエーションが出力されました。")
    print(f"出力ディレクトリ: {output_dir}")

def main():
    parser = argparse.ArgumentParser(description='GSplatで生成されたPLYファイルをオリジナルの3DGSと互換性のある形式に変換')
    parser.add_argument('input', help='入力PLYファイルのパス')
    parser.add_argument('--output', '-o', help='出力PLYファイルのパス (デフォルト: input_converted.ply)')
    parser.add_argument('--try-all', '-t', action='store_true', help='すべての変換バリエーションを試す')
    parser.add_argument('--output-dir', '-d', default='variants', help='複数のバリエーション用の出力ディレクトリ')
    
    # 座標系変換オプション
    parser.add_argument('--swap-axes', action='store_true', help='座標軸を入れ替える')
    parser.add_argument('--swap-mode', choices=['y_up_to_z_up', 'z_up_to_y_up'], default='y_up_to_z_up',
                        help='座標系変換のモード')
    
    # 色彩変換オプション
    parser.add_argument('--color-conversion', action='store_true', help='カラースペース変換を行う')
    parser.add_argument('--color-mode', choices=['linear_to_srgb', 'srgb_to_linear'], default='linear_to_srgb',
                        help='カラースペース変換のモード')
    
    # SH係数変換オプション
    parser.add_argument('--sh-conversion', choices=['gsplat_to_3dgs', '3dgs_to_gsplat', 'reorder_channels', 'swap_channels', 'normalize'],
                       default='gsplat_to_3dgs', help='球面調和関数係数の変換方法')
    
    # クォータニオン変換オプション
    parser.add_argument('--quaternion-conversion', action='store_true', help='クォータニオン規約変換を行う')
    parser.add_argument('--quat-mode', choices=['wxyz_to_xyzw', 'xyzw_to_wxyz', 'conjugate'], default='wxyz_to_xyzw',
                        help='クォータニオン変換のモード')
    
    # スケーリングオプション
    parser.add_argument('--scale-factor', type=float, default=1.0, help='スケールの調整係数')
    parser.add_argument('--opacity-scale', type=float, default=1.0, help='不透明度の調整係数')
    
    args = parser.parse_args()
    
    # 入力ファイルの存在を確認
    if not os.path.exists(args.input):
        print(f"エラー: 入力ファイル '{args.input}' が見つかりません。")
        return 1
    
    # 出力パスの設定
    if args.try_all:
        # 複数のバリエーションを試す
        try_all_conversions(args.input, args.output_dir)
    else:
        # 単一のファイル変換
        if not args.output:
            base, ext = os.path.splitext(args.input)
            output_path = f"{base}_converted{ext}"
        else:
            output_path = args.output
        
        # 変換オプションを設定
        options = {
            'swap_axes': args.swap_axes,
            'swap_mode': args.swap_mode,
            'color_space_conversion': args.color_conversion,
            'color_mode': args.color_mode,
            'sh_conversion': args.sh_conversion,
            'quaternion_convention': args.quaternion_conversion,
            'quat_mode': args.quat_mode,
            'scale_factor': args.scale_factor,
            'opacity_scale': args.opacity_scale
        }
        
        # 変換実行
        convert_gsplat_to_3dgs(args.input, output_path, options)
    
    return 0

input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

try_all_conversions(input_file, "test")


バリエーション: 基本変換（プロパティ名のみ）
変換開始: point_cloud_yuya.ply -> test\point_cloud_yuya_basic.ply
頂点数: 2000000
入力ファイルのプロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
DC成分: 3, REST成分: 45
スケール: 3, 回転: 4
推定SH次数: 3, 各チャンネルの係数数: 15
出力プロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2

変換完了: test\point_cloud_yuya_y_up_to_z_up.ply

バリエーション: 座標系変換（Z-up -> Y-up）
変換開始: point_cloud_yuya.ply -> test\point_cloud_yuya_z_up_to_y_up.ply
頂点数: 2000000
入力ファイルのプロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
DC成分: 3, REST成分: 45
スケール: 3, 回転: 4
推定SH次数: 3, 各チャンネルの係数数: 15
出力プロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_

変換完了: test\point_cloud_yuya_quat_xyzw_to_wxyz.ply

バリエーション: クォータニオン共役
変換開始: point_cloud_yuya.ply -> test\point_cloud_yuya_quat_conjugate.ply
頂点数: 2000000
入力ファイルのプロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
DC成分: 3, REST成分: 45
スケール: 3, 回転: 4
推定SH次数: 3, 各チャンネルの係数数: 15
出力プロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0',

変換完了: test\point_cloud_yuya_sh_flip_m_y_up_to_z_up.ply

バリエーション: SH係数反転 + 座標系変換（Z-up -> Y-up）
変換開始: point_cloud_yuya.ply -> test\point_cloud_yuya_sh_flip_m_z_up_to_y_up.ply
頂点数: 2000000
入力ファイルのプロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
DC成分: 3, REST成分: 45
スケール: 3, 回転: 4
推定SH次数: 3, 各チャンネルの係数数: 15
出力プロパティ: ['x', 'y', 

In [22]:
import os
import numpy as np
from plyfile import PlyData, PlyElement
import argparse
import sys

def rename_properties(input_path, output_path):
    """
    GSplatで生成されたPLYファイルのプロパティ名を3DGSと互換性のある形式に変換します
    
    Args:
        input_path: 入力PLYファイルのパス
        output_path: 出力PLYファイルのパス
    """
    print(f"ファイル変換開始: {input_path} -> {output_path}")
    
    # PLYファイルを読み込み
    try:
        ply_data = PlyData.read(input_path)
    except Exception as e:
        print(f"エラー: PLYファイルの読み込みに失敗しました - {e}")
        return False
    
    vertices = ply_data['vertex']
    
    # すべてのプロパティ名を取得
    property_names = [prop.name for prop in vertices.properties]
    print(f"元のプロパティ: {property_names}")
    
    # 新しいプロパティ名のマッピングを作成
    property_map = {}
    
    for name in property_names:
        # fdcX -> f_dc_X
        if name.startswith('fdc'):
            new_name = 'f_dc_' + name[3:]
        # f*rest*X -> f_rest_X
        elif name.startswith('f*rest*'):
            new_name = 'f_rest_' + name[7:]
        # scale*X -> scale_X
        elif name.startswith('scale*'):
            new_name = 'scale_' + name[6:]
        # rot*X -> rot_X
        elif name.startswith('rot*'):
            new_name = 'rot_' + name[4:]
        else:
            new_name = name
        
        property_map[name] = new_name
    
    # 各属性のデータを取得
    data_dict = {}
    for old_name, new_name in property_map.items():
        data_dict[new_name] = vertices[old_name]
    
    # 新しいデータ型を定義
    dtype_full = [(property_map[name], 'f4') for name in property_names]
    
    # 新しいデータ配列を作成
    elements = np.empty(len(vertices), dtype=dtype_full)
    
    # データを配列に格納
    for old_name, new_name in property_map.items():
        elements[new_name] = data_dict[new_name]
    
    # PLYファイルとして保存
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(output_path)
    
    print(f"変換完了: {input_path} -> {output_path}")
    print(f"新しいプロパティ: {[property_map[name] for name in property_names]}")
    return True

def reorder_sh_coefficients(input_path, output_path):
    """
    PLYファイルの球面調和関数の係数順序を変更して互換性を確保します
    
    Args:
        input_path: 入力PLYファイルのパス
        output_path: 出力PLYファイルのパス
    """
    print(f"SH係数の順序変換開始: {input_path} -> {output_path}")
    
    # PLYファイルを読み込み
    try:
        ply_data = PlyData.read(input_path)
    except Exception as e:
        print(f"エラー: PLYファイルの読み込みに失敗しました - {e}")
        return False
    
    vertices = ply_data['vertex']
    
    # すべてのプロパティ名を取得
    property_names = [prop.name for prop in vertices.properties]
    
    # REST成分のプロパティを抽出
    rest_props = [prop for prop in property_names if prop.startswith('f_rest_')]
    rest_indices = [int(prop.split('_')[-1]) for prop in rest_props]
    
    # SH次数を推定
    rest_count = len(rest_props)
    if rest_count == 0:
        print("警告: REST成分が見つかりません。SH係数の順序変換はスキップします。")
        return rename_properties(input_path, output_path)
    
    sh_degree = int(np.sqrt(rest_count / 3 + 1) - 1)
    print(f"推定SH次数: {sh_degree} (REST成分数: {rest_count})")
    
    # データの抽出
    data_dict = {}
    for prop in property_names:
        if not prop.startswith('f_rest_'):
            data_dict[prop] = vertices[prop]
    
    # REST成分データを抽出
    rest_values = np.zeros((len(vertices), rest_count))
    for i, prop in enumerate(sorted(rest_props, key=lambda p: int(p.split('_')[-1]))):
        rest_values[:, i] = vertices[prop]
    
    # 各チャンネルごとのREST成分の数
    rest_per_channel = rest_count // 3
    
    # 順序変換を適用したデータ
    reordered_rest = np.zeros_like(rest_values)
    
    # 球面調和関数の順序を変換
    for channel in range(3):  # RGB
        channel_offset = channel * rest_per_channel
        
        for l in range(1, sh_degree + 1):
            # 各次数における係数の開始インデックスと数
            degree_start = l * l - 1
            degree_size = 2 * l + 1
            
            for m_idx in range(degree_size):
                # GSplatでのインデックス（順序が異なる可能性）
                gsplat_idx = channel_offset + degree_start + m_idx
                
                # 3DGSでのインデックス
                # この部分は実際の3DGSとGSplatの実装に合わせて調整が必要
                # ここでは例としてm値の符号を反転させる変換を適用
                m = m_idx - l  # -l から l までのm値
                if m < 0:
                    m = -m  # m値の符号を反転
                
                threeDGS_idx = channel_offset + degree_start + (m + l)
                
                if gsplat_idx < rest_count and threeDGS_idx < rest_count:
                    reordered_rest[:, threeDGS_idx] = rest_values[:, gsplat_idx]
    
    # 変換後のデータを保存
    for i, prop in enumerate(sorted(rest_props, key=lambda p: int(p.split('_')[-1]))):
        data_dict[prop] = reordered_rest[:, i]
    
    # 新しいデータ型を定義
    dtype_full = [(name, 'f4') for name in property_names]
    
    # 新しいデータ配列を作成
    elements = np.empty(len(vertices), dtype=dtype_full)
    
    # データを配列に格納
    for name in property_names:
        elements[name] = data_dict[name]
    
    # PLYファイルとして保存
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(output_path)
    
    print(f"SH係数の順序変換完了: {input_path} -> {output_path}")
    return True

def try_many_variations(input_path, output_dir):
    """
    様々なSH係数の順序変換パターンを試みる関数
    
    Args:
        input_path: 入力PLYファイルのパス
        output_dir: 出力ディレクトリ
    """
    # 一時的に名前変換のみを適用したファイルを作成
    base_name = os.path.splitext(os.path.basename(input_path))[0]
    renamed_path = os.path.join(output_dir, f"{base_name}_renamed.ply")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # プロパティ名の変換
    if not rename_properties(input_path, renamed_path):
        print("エラー: プロパティ名の変換に失敗しました。")
        return
    
    # PLYファイルを読み込み
    try:
        ply_data = PlyData.read(renamed_path)
    except Exception as e:
        print(f"エラー: PLYファイルの読み込みに失敗しました - {e}")
        return
    
    vertices = ply_data['vertex']
    
    # すべてのプロパティ名を取得
    property_names = [prop.name for prop in vertices.properties]
    
    # REST成分のプロパティを抽出
    rest_props = [prop for prop in property_names if prop.startswith('f_rest_')]
    
    # SH次数を推定
    rest_count = len(rest_props)
    if rest_count == 0:
        print("警告: REST成分が見つかりません。変換を終了します。")
        return
    
    sh_degree = int(np.sqrt(rest_count / 3 + 1) - 1)
    print(f"推定SH次数: {sh_degree} (REST成分数: {rest_count})")
    
    # 各チャンネルごとのREST成分の数
    rest_per_channel = rest_count // 3
    
    # 変換パターンを定義
    patterns = []
    
    # パターン1: デフォルト（変換なし）
    patterns.append({
        'name': 'default',
        'description': 'デフォルト（変換なし）',
        'transform': lambda values: values.copy()
    })
    
    # パターン2: 各チャンネル内で完全に逆順
    patterns.append({
        'name': 'reverse_channel',
        'description': '各チャンネル内で逆順',
        'transform': lambda values: transform_reverse_channel(values, rest_per_channel)
    })
    
    # パターン3: 各次数内で逆順
    patterns.append({
        'name': 'reverse_degree',
        'description': '各次数内で逆順',
        'transform': lambda values: transform_reverse_degree(values, sh_degree, rest_per_channel)
    })
    
    # パターン4: m値の符号を反転
    patterns.append({
        'name': 'flip_m_sign',
        'description': 'm値の符号を反転',
        'transform': lambda values: transform_flip_m_sign(values, sh_degree, rest_per_channel)
    })
    
    # パターン5: 各次数内でm値の順序を反転
    patterns.append({
        'name': 'reverse_m',
        'description': '各次数内でm値の順序を反転',
        'transform': lambda values: transform_reverse_m(values, sh_degree, rest_per_channel)
    })
    
    # パターン6: Zonal Harmonics <-> Spherical Harmonics変換
    patterns.append({
        'name': 'ylm_to_zonal',
        'description': 'Ylm順からZonal Harmonics順に変換',
        'transform': lambda values: transform_ylm_to_zonal(values, sh_degree, rest_per_channel)
    })
    
    # データの抽出
    rest_values = np.zeros((len(vertices), rest_count))
    for i, prop in enumerate(sorted(rest_props, key=lambda p: int(p.split('_')[-1]))):
        rest_values[:, i] = vertices[prop]
    
    # 各パターンを適用
    for pattern in patterns:
        output_path = os.path.join(output_dir, f"{base_name}_{pattern['name']}.ply")
        print(f"\nパターン適用: {pattern['description']} -> {output_path}")
        
        # 変換を適用
        transformed = pattern['transform'](rest_values)
        
        # データ辞書を作成
        data_dict = {}
        for prop in property_names:
            if not prop.startswith('f_rest_'):
                data_dict[prop] = vertices[prop]
        
        # 変換したREST成分を追加
        for i, prop in enumerate(sorted(rest_props, key=lambda p: int(p.split('_')[-1]))):
            data_dict[prop] = transformed[:, i]
        
        # 新しいデータ型を定義
        dtype_full = [(name, 'f4') for name in property_names]
        
        # 新しいデータ配列を作成
        elements = np.empty(len(vertices), dtype=dtype_full)
        
        # データを配列に格納
        for name in property_names:
            elements[name] = data_dict[name]
        
        # PLYファイルとして保存
        el = PlyElement.describe(elements, 'vertex')
        PlyData([el]).write(output_path)
    
    print(f"\n合計 {len(patterns)} パターンの変換が完了しました。")
    print(f"変換結果の保存先: {output_dir}")

def transform_reverse_channel(values, rest_per_channel):
    """各チャンネル内で逆順に並べ替える"""
    result = values.copy()
    channels = values.shape[1] // rest_per_channel
    
    for c in range(channels):
        start = c * rest_per_channel
        end = (c + 1) * rest_per_channel
        
        for i in range(rest_per_channel):
            rev_i = rest_per_channel - 1 - i
            result[:, start + i] = values[:, start + rev_i]
    
    return result

def transform_reverse_degree(values, sh_degree, rest_per_channel):
    """各次数内で逆順に並べ替える"""
    result = values.copy()
    channels = values.shape[1] // rest_per_channel
    
    for c in range(channels):
        channel_offset = c * rest_per_channel
        
        for l in range(1, sh_degree + 1):
            degree_start = l * l - 1
            degree_size = 2 * l + 1
            
            for i in range(degree_size):
                rev_i = degree_size - 1 - i
                idx1 = channel_offset + degree_start + i
                idx2 = channel_offset + degree_start + rev_i
                
                if idx1 < values.shape[1] and idx2 < values.shape[1]:
                    result[:, idx1] = values[:, idx2]
    
    return result

def transform_flip_m_sign(values, sh_degree, rest_per_channel):
    """m値の符号を反転させる"""
    result = values.copy()
    channels = values.shape[1] // rest_per_channel
    
    for c in range(channels):
        channel_offset = c * rest_per_channel
        
        for l in range(1, sh_degree + 1):
            degree_start = l * l - 1
            
            for m in range(-l, l + 1):
                m_idx = m + l  # -l から l までを 0 から 2l までのインデックスに変換
                
                # m値の符号を反転（m=0は変更なし）
                flipped_m = -m if m != 0 else 0
                flipped_m_idx = flipped_m + l
                
                idx1 = channel_offset + degree_start + m_idx
                idx2 = channel_offset + degree_start + flipped_m_idx
                
                if idx1 < values.shape[1] and idx2 < values.shape[1]:
                    result[:, idx1] = values[:, idx2]
    
    return result

def transform_reverse_m(values, sh_degree, rest_per_channel):
    """各次数内でm値の順序を反転"""
    result = values.copy()
    channels = values.shape[1] // rest_per_channel
    
    for c in range(channels):
        channel_offset = c * rest_per_channel
        
        for l in range(1, sh_degree + 1):
            degree_start = l * l - 1
            
            for m in range(-l, l + 1):
                m_idx = m + l
                
                # m値の順序を反転（-l→l, -(l-1)→l-1, ..., l→-l）
                reversed_m = -l + (2 * l - m_idx)
                reversed_m_idx = reversed_m + l
                
                idx1 = channel_offset + degree_start + m_idx
                idx2 = channel_offset + degree_start + reversed_m_idx
                
                if idx1 < values.shape[1] and idx2 < values.shape[1]:
                    result[:, idx1] = values[:, idx2]
    
    return result

def transform_ylm_to_zonal(values, sh_degree, rest_per_channel):
    """Ylm順からZonal Harmonics順に変換"""
    result = values.copy()
    channels = values.shape[1] // rest_per_channel
    
    for c in range(channels):
        channel_offset = c * rest_per_channel
        
        for l in range(1, sh_degree + 1):
            degree_start_ylm = l * l - 1
            
            for m in range(-l, l + 1):
                # Ylm順でのインデックス
                m_idx = m + l
                ylm_idx = channel_offset + degree_start_ylm + m_idx
                
                # Zonal Harmonics順でのインデックス計算
                zonal_idx = channel_offset
                if m == 0:  # m=0
                    zonal_idx += l * l
                elif m > 0:  # m>0
                    zonal_idx += l * l + 2 * m
                else:  # m<0
                    zonal_idx += l * l + 2 * abs(m) - 1
                
                if ylm_idx < values.shape[1] and zonal_idx < values.shape[1]:
                    result[:, zonal_idx] = values[:, ylm_idx]
    
    return result

"""
def main():
    parser = argparse.ArgumentParser(description='GSplatで生成されたPLYファイルをオリジナルの3DGSと互換性のある形式に変換します')
    parser.add_argument('input', help='入力PLYファイルのパス')
    parser.add_argument('--output', '-o', help='出力PLYファイルのパス (デフォルト: input_converted.ply)')
    parser.add_argument('--rename-only', '-r', action='store_true', help='プロパティ名の変換のみを行い、SH係数の順序変換はスキップする')
    parser.add_argument('--try-all', '-t', action='store_true', help='様々なSH係数の順序変換パターンを試みる')
    parser.add_argument('--output-dir', '-d', default='sh_variations', help='複数変換時の出力ディレクトリ')
    
    args = parser.parse_args()
    
    # 入力ファイルの存在確認
    if not os.path.exists(args.input):
        print(f"エラー: 入力ファイル '{args.input}' が見つかりません。")
        sys.exit(1)
    
    # 出力パスの設定
    if not args.output:
        base, ext = os.path.splitext(args.input)
        output_path = f"{base}_converted{ext}"
    else:
        output_path = args.output
    
    # 変換処理
    if args.try_all:
        try_many_variations(args.input, args.output_dir)
    elif args.rename_only:
        rename_properties(args.input, output_path)
    else:
        reorder_sh_coefficients(args.input, output_path)
"""

    
input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

try_many_variations(input_file, "test")

ファイル変換開始: point_cloud_yuya.ply -> test\point_cloud_yuya_renamed.ply
元のプロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
変換完了: point_cloud_yuya.ply -> test\point_cloud_yuya_renamed.ply
新しいプロパティ: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_res

In [27]:
import os
import numpy as np
from plyfile import PlyData, PlyElement

def fix_ply_format(input_path, output_path):
    """
    GSplatで生成されたPLYファイルをSuperSplatと互換性のある形式に変換します
    
    Args:
        input_path: 入力PLYファイルのパス
        output_path: 出力PLYファイルのパス
    """
    # PLYファイルを読み込み
    ply_data = PlyData.read(input_path)
    vertices = ply_data['vertex']
    
    # すべてのプロパティ名を取得
    property_names = [prop.name for prop in vertices.properties]
    
    # 新しいプロパティ名のリストを作成
    new_property_names = []
    dc_count = 0
    rest_count = 0
    scale_count = 0
    rot_count = 0
    
    for name in property_names:
        if name.startswith('fdc'):
            new_name = f'f_dc_{dc_count}'
            dc_count += 1
        elif name.startswith('f*rest*'):
            new_name = f'f_rest_{rest_count}'
            rest_count += 1
        elif name == 'opacity':
            new_name = 'opacity'
        elif name.startswith('scale*'):
            new_name = f'scale_{scale_count}'
            scale_count += 1
        elif name.startswith('rot*'):
            new_name = f'rot_{rot_count}'
            rot_count += 1
        else:
            new_name = name
        new_property_names.append(new_name)
    
    # データの抽出
    data = np.zeros((len(vertices), len(property_names)), dtype=np.float32)
    for i, prop in enumerate(property_names):
        data[:, i] = vertices[prop]
    
    # 新しい頂点要素を作成
    dtype_full = [(new_name, 'f4') for new_name in new_property_names]
    elements = np.empty(len(vertices), dtype=dtype_full)
    elements[:] = list(map(tuple, data))
    
    # PLYファイルとして保存
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(output_path)
    
    print(f"変換完了: {input_path} -> {output_path}")
    print(f"属性名の変更:")
    for old, new in zip(property_names, new_property_names):
        if old != new:
            print(f"  {old} -> {new}")


    
input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

fix_ply_format(input_file, output_file) # デフォルトはバイナリ出力    

変換完了: point_cloud_yuya.ply -> point_cloud_yuya_converted.ply
属性名の変更:


In [ ]:
import os
import struct
import numpy as np
from plyfile import PlyData, PlyElement
import argparse

def convert_gsplat_to_supersplat(input_path, output_path):
    """
    GSplatで生成したPLYファイルをSupersplatで正常に表示できる形式に変換します。
    
    Args:
        input_path: 入力PLYファイルのパス
        output_path: 出力PLYファイルのパス
    """
    print(f"Converting {input_path} to Supersplat format...")
    
    # PLYファイルを読み込む
    try:
        ply_data = PlyData.read(input_path)
        vertex_data = ply_data['vertex']
    except Exception as e:
        print(f"Error reading PLY file: {e}")
        return False
    
    # 属性情報を抽出
    means = np.vstack([vertex_data['x'], vertex_data['y'], vertex_data['z']]).T
    
    # 球面調和関数係数を抽出
    f_dc_properties = [prop for prop in vertex_data.properties if prop.name.startswith('f_dc_')]
    f_rest_properties = [prop for prop in vertex_data.properties if prop.name.startswith('f_rest_')]
    
    # 球面調和関数の配列を作成
    sh0 = np.zeros((len(vertex_data), len(f_dc_properties)))
    shN = np.zeros((len(vertex_data), len(f_rest_properties)))
    
    # 球面調和関数の値を設定
    for i, prop in enumerate(f_dc_properties):
        sh0[:, i] = vertex_data[prop.name]
    
    for i, prop in enumerate(f_rest_properties):
        shN[:, i] = vertex_data[prop.name]
    
    # スケール、回転、不透明度を抽出
    scale_properties = [prop for prop in vertex_data.properties if prop.name.startswith('scale_')]
    rot_properties = [prop for prop in vertex_data.properties if prop.name.startswith('rot_')]
    
    scales = np.zeros((len(vertex_data), len(scale_properties)))
    quats = np.zeros((len(vertex_data), len(rot_properties)))
    opacities = vertex_data['opacity']
    
    for i, prop in enumerate(scale_properties):
        scales[:, i] = vertex_data[prop.name]
    
    for i, prop in enumerate(rot_properties):
        quats[:, i] = vertex_data[prop.name]
    
    # 無効なデータを検出するマスクを作成
    invalid_mask = (
        np.isnan(means).any(axis=1)
        | np.isinf(means).any(axis=1)
        | np.isnan(scales).any(axis=1)
        | np.isinf(scales).any(axis=1)
        | np.isnan(quats).any(axis=1)
        | np.isinf(quats).any(axis=1)
        | np.isnan(opacities)
        | np.isinf(opacities)
        | np.isnan(sh0).any(axis=1)
        | np.isinf(sh0).any(axis=1)
        | np.isnan(shN).any(axis=1)
        | np.isinf(shN).any(axis=1)
    )
    
    # 無効なデータを除外
    means = means[~invalid_mask]
    scales = scales[~invalid_mask]
    quats = quats[~invalid_mask]
    opacities = opacities[~invalid_mask]
    sh0 = sh0[~invalid_mask]
    shN = shN[~invalid_mask]
    
    num_points = means.shape[0]
    print(f"Total valid points: {num_points}")
    print(f"Invalid points removed: {np.sum(invalid_mask)}")
    
    # 出力ディレクトリが存在しない場合は作成
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    
    # カラー情報（f_dc_0-2）を正規化
    if sh0.shape[1] >= 3:
        # 値の範囲をチェック
        min_values = np.min(sh0[:, :3], axis=0)
        max_values = np.max(sh0[:, :3], axis=0)
        print(f"Color coefficient ranges before conversion:")
        print(f"  Min values: {min_values}")
        print(f"  Max values: {max_values}")
        
        # カラー値を正規化する必要があるかどうかを判断
        if np.min(min_values) < -0.01 or np.max(max_values) > 1.01:
            # 値が標準範囲外なら変換
            sh0_converted = sh0.copy()
            sh0_converted[:, :3] = sh0[:, :3] * 0.282094791773878 + 0.5
            sh0_converted[:, :3] = np.clip(sh0_converted[:, :3], 0, 1)
            print("Color coefficients were converted to standard range [0,1]")
            
            # 変換後の範囲を表示
            min_values = np.min(sh0_converted[:, :3], axis=0)
            max_values = np.max(sh0_converted[:, :3], axis=0)
            print(f"Color coefficient ranges after conversion:")
            print(f"  Min values: {min_values}")
            print(f"  Max values: {max_values}")
        else:
            # 既に適切な範囲内なら変換不要
            sh0_converted = sh0
            print("Color coefficients were already in standard range [0,1], no conversion needed")
    else:
        sh0_converted = sh0
    
    # PLYファイルを書き出す（Supersplat形式）
    try:
        with open(output_path, "wb") as f:
            # PLYヘッダーを書き込む
            f.write(b"ply\n")
            f.write(b"format binary_little_endian 1.0\n")
            f.write(f"element vertex {num_points}\n".encode())
            f.write(b"property float x\n")
            f.write(b"property float y\n")
            f.write(b"property float z\n")
            f.write(b"property float nx\n")
            f.write(b"property float ny\n")
            f.write(b"property float nz\n")
            
            # 球面調和関数のプロパティを書き込む
            for j in range(sh0.shape[1]):
                f.write(f"property float f_dc_{j}\n".encode())
            
            for j in range(shN.shape[1]):
                f.write(f"property float f_rest_{j}\n".encode())
            
            # その他のプロパティを書き込む
            f.write(b"property float opacity\n")
            
            for i in range(scales.shape[1]):
                f.write(f"property float scale_{i}\n".encode())
            
            for i in range(quats.shape[1]):
                f.write(f"property float rot_{i}\n".encode())
            
            f.write(b"end_header\n")
            
            # 頂点データを書き込む
            for i in range(num_points):
                f.write(struct.pack("<fff", *means[i]))  # x, y, z
                f.write(struct.pack("<fff", 0, 0, 0))    # nx, ny, nz (zeros)
                
                # 球面調和関数を書き込む
                for j in range(sh0_converted.shape[1]):
                    f.write(struct.pack("<f", sh0_converted[i, j]))
                
                for j in range(shN.shape[1]):
                    f.write(struct.pack("<f", shN[i, j]))
                
                # 不透明度を書き込む
                f.write(struct.pack("<f", opacities[i]))
                
                # スケールと四元数を書き込む
                for j in range(scales.shape[1]):
                    f.write(struct.pack("<f", scales[i, j]))
                
                for j in range(quats.shape[1]):
                    f.write(struct.pack("<f", quats[i, j]))
        
        print(f"Converted file saved to {output_path}")
        return True
    
    except Exception as e:
        print(f"Error writing PLY file: {e}")
        return False

def check_ply_info(file_path):
    """
    PLYファイルの詳細情報を表示します。
    """
    try:
        ply_data = PlyData.read(file_path)
        
        print(f"\nPLY File Information for: {file_path}")
        print(f"Format: {ply_data.fmt}")
        print(f"Version: {ply_data.version}")
        
        for element in ply_data.elements:
            print(f"\nElement: {element.name}")
            print(f"Count: {len(element)}")
            
            print("Properties:")
            for prop in element.properties:
                print(f"  - {prop.name}: {prop.val_dtype}")
                
                # 最初の数個の値を表示
                if element.name == 'vertex' and len(element) > 0:
                    values = element[prop.name][:5]
                    if isinstance(values, np.ndarray) and values.size > 5:
                        values = values[:5]
                    print(f"    Sample values: {values}")
                    
                    # f_dc_0-2が色情報なら、範囲を確認
                    if prop.name in ['f_dc_0', 'f_dc_1', 'f_dc_2']:
                        all_values = element[prop.name]
                        print(f"    Range: {np.min(all_values):.4f} to {np.max(all_values):.4f}")
        
        return True
    except Exception as e:
        print(f"Error reading PLY file: {e}")
  
            
input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_ply_to_supersplat_format(input_file, output_file) # デフォルトはバイナリ出力
            


In [9]:
import os
import struct
import numpy as np
from plyfile import PlyData, PlyElement
import argparse

def convert_ply_to_supersplat_format(input_path, output_path):
    """
    Convert a PLY file from GSplat format to Supersplat compatible format.
    
    Args:
        input_path: Path to the input PLY file
        output_path: Path to save the converted PLY file
    """
    print(f"Converting {input_path} to Supersplat format...")
    
    # Load the PLY file
    ply_data = PlyData.read(input_path)
    vertex_data = ply_data['vertex']
    
    # Extract all properties
    means = np.vstack([vertex_data['x'], vertex_data['y'], vertex_data['z']]).T
    
    # Extract spherical harmonics
    f_dc_properties = [prop for prop in vertex_data.properties if prop.name.startswith('f_dc_')]
    f_rest_properties = [prop for prop in vertex_data.properties if prop.name.startswith('f_rest_')]
    
    # Create arrays for spherical harmonics
    sh0 = np.zeros((len(vertex_data), len(f_dc_properties)))
    shN = np.zeros((len(vertex_data), len(f_rest_properties)))
    
    # Fill spherical harmonics arrays
    for i, prop in enumerate(f_dc_properties):
        sh0[:, i] = vertex_data[prop.name]
    
    for i, prop in enumerate(f_rest_properties):
        shN[:, i] = vertex_data[prop.name]
    
    # Extract scales, rotations and opacity
    scale_properties = [prop for prop in vertex_data.properties if prop.name.startswith('scale_')]
    rot_properties = [prop for prop in vertex_data.properties if prop.name.startswith('rot_')]
    
    scales = np.zeros((len(vertex_data), len(scale_properties)))
    quats = np.zeros((len(vertex_data), len(rot_properties)))
    opacities = vertex_data['opacity']
    
    for i, prop in enumerate(scale_properties):
        scales[:, i] = vertex_data[prop.name]
    
    for i, prop in enumerate(rot_properties):
        quats[:, i] = vertex_data[prop.name]
    
    # Create a mask to identify rows with NaN or Inf
    invalid_mask = (
        np.isnan(means).any(axis=1)
        | np.isinf(means).any(axis=1)
        | np.isnan(scales).any(axis=1)
        | np.isinf(scales).any(axis=1)
        | np.isnan(quats).any(axis=1)
        | np.isinf(quats).any(axis=1)
        | np.isnan(opacities)
        | np.isinf(opacities)
        | np.isnan(sh0).any(axis=1)
        | np.isinf(sh0).any(axis=1)
        | np.isnan(shN).any(axis=1)
        | np.isinf(shN).any(axis=1)
    )
    
    # Filter out invalid data
    means = means[~invalid_mask]
    scales = scales[~invalid_mask]
    quats = quats[~invalid_mask]
    opacities = opacities[~invalid_mask]
    sh0 = sh0[~invalid_mask]
    shN = shN[~invalid_mask]
    
    num_points = means.shape[0]
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    
    # Write to new PLY file in Supersplat format
    with open(output_path, "wb") as f:
        # Write PLY header
        f.write(b"ply\n")
        f.write(b"format binary_little_endian 1.0\n")
        f.write(f"element vertex {num_points}\n".encode())
        f.write(b"property float x\n")
        f.write(b"property float y\n")
        f.write(b"property float z\n")
        f.write(b"property float nx\n")
        f.write(b"property float ny\n")
        f.write(b"property float nz\n")
        
        # Write spherical harmonics properties
        for j in range(sh0.shape[1]):
            f.write(f"property float f_dc_{j}\n".encode())
        
        for j in range(shN.shape[1]):
            f.write(f"property float f_rest_{j}\n".encode())
        
        # Write other properties
        f.write(b"property float opacity\n")
        
        for i in range(scales.shape[1]):
            f.write(f"property float scale_{i}\n".encode())
        
        for i in range(quats.shape[1]):
            f.write(f"property float rot_{i}\n".encode())
        
        f.write(b"end_header\n")
        
        # Write vertex data
        for i in range(num_points):
            f.write(struct.pack("<fff", *means[i]))  # x, y, z
            f.write(struct.pack("<fff", 0, 0, 0))    # nx, ny, nz (zeros)
            
            # Write spherical harmonics
            for j in range(sh0.shape[1]):
                f.write(struct.pack("<f", sh0[i, j]))
            
            for j in range(shN.shape[1]):
                f.write(struct.pack("<f", shN[i, j]))
            
            # Write opacity
            f.write(struct.pack("<f", opacities[i]))
            
            # Write scales and quaternions
            for j in range(scales.shape[1]):
                f.write(struct.pack("<f", scales[i, j]))
            
            for j in range(quats.shape[1]):
                f.write(struct.pack("<f", quats[i, j]))
    
    print(f"Converted file saved to {output_path}")
    print(f"Total points: {num_points}")

input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_ply_to_supersplat_format(input_file, output_file) # デフォルトはバイナリ出力

Converting point_cloud_yuya.ply to Supersplat format...
Converted file saved to point_cloud_yuya_converted.ply
Total points: 2000000


In [5]:
import numpy as np
import argparse
import os
import struct
import re
from plyfile import PlyData, PlyElement

def convert_to_supersplat(input_file, output_file, binary=True):
    """
    カスタム実装から出力されたPLYファイルをSuperSplat互換のPLYファイルに変換する
    
    Args:
        input_file (str): 入力カスタムPLYファイルへのパス
        output_file (str): 出力SuperSplat PLYファイルへのパス
        binary (bool): バイナリ形式で出力するかどうか（デフォルトはTrue）
    """
    print(f"入力ファイル: {input_file}")
    print(f"出力ファイル: {output_file}")
    print(f"出力形式: {'バイナリ' if binary else 'ASCII'}")
    
    # PLYファイルを読み込む
    ply_data = PlyData.read(input_file)
    vertex_data = ply_data['vertex']
    
    # データ数を取得
    num_points = len(vertex_data)
    print(f"頂点数: {num_points}")
    
    # 利用可能な属性名のリストを取得
    property_names = [prop.name for prop in vertex_data.properties]
    print(f"利用可能な属性: {', '.join(property_names[:10])}...")
    
    # 位置座標（x, y, z）を取得
    positions = np.zeros((num_points, 3))
    for i, coord in enumerate(['x', 'y', 'z']):
        if coord in property_names:
            positions[:, i] = vertex_data[coord]
    
    # 法線情報（nx, ny, nz）を取得
    normals = np.zeros((num_points, 3))
    for i, coord in enumerate(['nx', 'ny', 'nz']):
        if coord in property_names:
            normals[:, i] = vertex_data[coord]
    
    # SH係数の抽出
    # 正規表現を使用してプロパティ名からインデックスを抽出
    # 両方の形式（fdc0とf_dc_0）をサポート
    
    # DC係数（f_dc_0, f_dc_1, f_dc_2）
    dc_pattern = re.compile(r'f_?dc_?(\d+)')
    fdc_keys = []
    fdc_indices = []
    
    for name in property_names:
        match = dc_pattern.match(name)
        if match:
            fdc_keys.append(name)
            # インデックスを抽出して整数に変換
            try:
                idx = int(match.group(1))
                fdc_indices.append((name, idx))
            except ValueError:
                print(f"警告: プロパティ名 '{name}' から有効なインデックスを抽出できませんでした。")
    
    # インデックスでソート
    fdc_indices.sort(key=lambda x: x[1])
    sorted_fdc_keys = [item[0] for item in fdc_indices]
    
    # DC係数を取得
    f_dc = np.zeros((num_points, max(3, len(sorted_fdc_keys))))  # 最低3つ確保
    for i, key in enumerate(sorted_fdc_keys):
        if i < f_dc.shape[1]:
            f_dc[:, i] = vertex_data[key]
    
    # 残りのSH係数（f_rest_0, f_rest_1, ...）
    rest_pattern = re.compile(r'f_?rest_?(\d+)')
    frest_keys = []
    frest_indices = []
    
    for name in property_names:
        match = rest_pattern.match(name)
        if match:
            frest_keys.append(name)
            # インデックスを抽出して整数に変換
            try:
                idx = int(match.group(1))
                frest_indices.append((name, idx))
            except ValueError:
                print(f"警告: プロパティ名 '{name}' から有効なインデックスを抽出できませんでした。")
    
    # インデックスでソート
    frest_indices.sort(key=lambda x: x[1])
    sorted_frest_keys = [item[0] for item in frest_indices]
    
    # SuperSplatの期待する45個のRest係数
    num_rest_required = 45
    f_rest = np.zeros((num_points, num_rest_required))
    
    # Rest係数を設定
    for i, key in enumerate(sorted_frest_keys):
        if i < num_rest_required:
            f_rest[:, i] = vertex_data[key]
    
    # 不透明度を取得
    opacities = np.ones(num_points)  # デフォルト値
    if 'opacity' in property_names:
        opacities = vertex_data['opacity']
    
    # スケール情報を取得
    scale_pattern = re.compile(r'scale_?(\d+)')
    scale_indices = []
    
    for name in property_names:
        match = scale_pattern.match(name)
        if match:
            try:
                idx = int(match.group(1))
                scale_indices.append((name, idx))
            except ValueError:
                print(f"警告: プロパティ名 '{name}' から有効なインデックスを抽出できませんでした。")
    
    # インデックスでソート
    scale_indices.sort(key=lambda x: x[1])
    sorted_scale_keys = [item[0] for item in scale_indices]
    
    scales = np.ones((num_points, 3)) * 0.01  # デフォルト値
    for i, key in enumerate(sorted_scale_keys):
        if i < 3:
            scales[:, i] = vertex_data[key]
    
    # 回転情報（四元数）を取得
    rot_pattern = re.compile(r'rot_?(\d+)')
    rot_indices = []
    
    for name in property_names:
        match = rot_pattern.match(name)
        if match:
            try:
                idx = int(match.group(1))
                rot_indices.append((name, idx))
            except ValueError:
                print(f"警告: プロパティ名 '{name}' から有効なインデックスを抽出できませんでした。")
    
    # インデックスでソート
    rot_indices.sort(key=lambda x: x[1])
    sorted_rot_keys = [item[0] for item in rot_indices]
    
    rotations = np.zeros((num_points, 4))
    rotations[:, 0] = 1.0  # 単位四元数（w=1, x=0, y=0, z=0）
    for i, key in enumerate(sorted_rot_keys):
        if i < 4:
            rotations[:, i] = vertex_data[key]
    
    # DC係数の確認と調整
    # カラー情報として使用される最初の3つのDC係数を確認
    if f_dc.shape[1] >= 3:
        print(f"DC係数の統計:")
        for i in range(3):
            dc_min = f_dc[:, i].min()
            dc_max = f_dc[:, i].max()
            dc_mean = f_dc[:, i].mean()
            print(f"  f_dc_{i}: 最小値={dc_min:.6f}, 最大値={dc_max:.6f}, 平均値={dc_mean:.6f}")
        
        # DC係数が極端に小さいか大きい場合、参照実装のフォーマットに合わせて調整
        if np.max(np.abs(f_dc[:, :3])) > 10 or np.max(np.abs(f_dc[:, :3])) < 0.01:
            print("警告: DC係数の範囲が異常です。SuperSplat互換の範囲に調整します。")
            
            # 参照実装から逆算すると、(RGB値-0.5)/0.2820947917738781 という変換式が使われている
            # ここでは、DC係数をこの式に合わせて調整する
            # RGB値として0.2～0.8の範囲を想定し、それに対応するDC係数を生成
            scaled_dc = np.zeros_like(f_dc[:, :3])
            for i in range(3):
                # 0～1の範囲に正規化（必要に応じて調整）
                normalized = (f_dc[:, i] - np.min(f_dc[:, i])) / (np.max(f_dc[:, i]) - np.min(f_dc[:, i]) + 1e-10)
                # 0.2～0.8の範囲に拡張（より自然な色合い）
                rgb = 0.2 + normalized * 0.6
                # SuperSplat互換の変換式を適用
                scaled_dc[:, i] = (rgb - 0.5) / 0.2820947917738781
            
            # 調整後の統計を表示
            print("調整後のDC係数の統計:")
            for i in range(3):
                dc_min = scaled_dc[:, i].min()
                dc_max = scaled_dc[:, i].max()
                dc_mean = scaled_dc[:, i].mean()
                print(f"  f_dc_{i}: 最小値={dc_min:.6f}, 最大値={dc_max:.6f}, 平均値={dc_mean:.6f}")
            
            # 調整後の値を使用
            f_dc[:, :3] = scaled_dc
    
    # SuperSplat互換のPLYファイルを作成
    with open(output_file, 'wb' if binary else 'w') as f:
        # ヘッダー部分
        if binary:
            f.write(b"ply\n")
            f.write(b"format binary_little_endian 1.0\n")
            f.write(f"element vertex {num_points}\n".encode())
        else:
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write(f"element vertex {num_points}\n")
        
        # プロパティ定義
        properties = [
            "property float x",
            "property float y",
            "property float z",
            "property float nx",
            "property float ny",
            "property float nz",
            "property float f_dc_0",
            "property float f_dc_1",
            "property float f_dc_2"
        ]
        
        # f_rest_0 から f_rest_44 までのプロパティを追加
        for i in range(num_rest_required):
            properties.append(f"property float f_rest_{i}")
        
        # 残りのプロパティ（正確な順序を維持）
        properties.extend([
            "property float opacity",
            "property float scale_0",
            "property float scale_1",
            "property float scale_2",
            "property float rot_0",
            "property float rot_1",
            "property float rot_2",
            "property float rot_3"
        ])
        
        # プロパティをファイルに書き込む
        for prop in properties:
            if binary:
                f.write(f"{prop}\n".encode())
            else:
                f.write(f"{prop}\n")
        
        # ヘッダー終了
        if binary:
            f.write(b"end_header\n")
        else:
            f.write("end_header\n")
        
        # データの書き込み
        for i in range(num_points):
            if binary:
                # 位置（x, y, z）
                f.write(struct.pack("<fff", float(positions[i, 0]), float(positions[i, 1]), float(positions[i, 2])))
                
                # 法線（nx, ny, nz）
                f.write(struct.pack("<fff", float(normals[i, 0]), float(normals[i, 1]), float(normals[i, 2])))
                
                # DC係数（f_dc_0, f_dc_1, f_dc_2）
                f.write(struct.pack("<fff", float(f_dc[i, 0]), float(f_dc[i, 1]), float(f_dc[i, 2])))
                
                # Rest係数（f_rest_0 から f_rest_44）
                for j in range(num_rest_required):
                    f.write(struct.pack("<f", float(f_rest[i, j])))
                
                # 不透明度
                f.write(struct.pack("<f", float(opacities[i])))
                
                # スケール
                f.write(struct.pack("<fff", float(scales[i, 0]), float(scales[i, 1]), float(scales[i, 2])))
                
                # 回転四元数
                f.write(struct.pack("<ffff", float(rotations[i, 0]), float(rotations[i, 1]), 
                                         float(rotations[i, 2]), float(rotations[i, 3])))
            else:
                # ASCII形式での書き込み
                line_parts = []
                
                # 位置（x, y, z）
                line_parts.extend([str(positions[i, 0]), str(positions[i, 1]), str(positions[i, 2])])
                
                # 法線（nx, ny, nz）
                line_parts.extend([str(normals[i, 0]), str(normals[i, 1]), str(normals[i, 2])])
                
                # DC係数（f_dc_0, f_dc_1, f_dc_2）
                line_parts.extend([str(f_dc[i, 0]), str(f_dc[i, 1]), str(f_dc[i, 2])])
                
                # Rest係数（f_rest_0 から f_rest_44）
                for j in range(num_rest_required):
                    line_parts.append(str(f_rest[i, j]))
                
                # 不透明度
                line_parts.append(str(opacities[i]))
                
                # スケール
                line_parts.extend([str(scales[i, 0]), str(scales[i, 1]), str(scales[i, 2])])
                
                # 回転四元数
                line_parts.extend([str(rotations[i, 0]), str(rotations[i, 1]), 
                                  str(rotations[i, 2]), str(rotations[i, 3])])
                
                # 1行としてまとめて書き込み
                f.write(" ".join(line_parts) + "\n")
    
    print(f"変換が完了しました。出力ファイル: {output_file}")

input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_to_supersplat(input_file, output_file) # デフォルトはバイナリ出力

入力ファイル: point_cloud_yuya.ply
出力ファイル: point_cloud_yuya_converted.ply
出力形式: バイナリ
頂点数: 2000000
利用可能な属性: x, y, z, nx, ny, nz, f_dc_0, f_dc_1, f_dc_2, f_rest_0...
DC係数の統計:
  f_dc_0: 最小値=-2.760195, 最大値=14.992920, 平均値=0.170825
  f_dc_1: 最小値=-2.772074, 最大値=14.173110, 平均値=0.202592
  f_dc_2: 最小値=-2.709437, 最大値=14.615707, 平均値=0.202587
警告: DC係数の範囲が異常です。SuperSplat互換の範囲に調整します。
調整後のDC係数の統計:
  f_dc_0: 最小値=-1.063472, 最大値=1.063472, 平均値=-0.712316
  f_dc_1: 最小値=-1.063472, 最大値=1.063472, 平均値=-0.690095
  f_dc_2: 最小値=-1.063472, 最大値=1.063472, 平均値=-0.705974
変換が完了しました。出力ファイル: point_cloud_yuya_converted.ply


In [2]:
import numpy as np
import argparse
import os
import struct
from plyfile import PlyData, PlyElement

def convert_custom_to_supersplat(input_file, output_file, binary=True, sh_scale=1.0):
    """
    カスタム実装から出力されたPLYファイルをSuperSplat互換のPLYファイルに変換する
    
    Args:
        input_file (str): 入力カスタムPLYファイルへのパス
        output_file (str): 出力SuperSplat PLYファイルへのパス
        binary (bool): バイナリ形式で出力するかどうか（デフォルトはTrue）
        sh_scale (float): SH係数のスケーリング係数（色の強度調整）
    """
    print(f"入力ファイル: {input_file}")
    print(f"出力ファイル: {output_file}")
    print(f"出力形式: {'バイナリ' if binary else 'ASCII'}")
    print(f"SH係数スケール: {sh_scale}")
    
    # PLYファイルを読み込む
    ply_data = PlyData.read(input_file)
    vertex_data = ply_data['vertex']
    
    # データ数を取得
    num_points = len(vertex_data)
    print(f"頂点数: {num_points}")
    
    # 利用可能な属性名のリストを取得
    property_names = [prop.name for prop in vertex_data.properties]
    print(f"利用可能な属性: {', '.join(property_names[:10])}...")
    
    # 名前マッピングの作成
    # カスタム実装の名前からSuperSplatの名前へのマッピング
    name_mapping = {}
    
    # 基本的な位置と法線のマッピング
    for name in ['x', 'y', 'z', 'nx', 'ny', 'nz']:
        name_mapping[name] = name
    
    # fdc -> f_dc_ のマッピング
    fdc_indices = []
    for name in property_names:
        if name.startswith('fdc'):
            index = name[3:]  # "fdc0" -> "0"
            name_mapping[name] = f'f_dc_{index}'
            fdc_indices.append(int(index))
    
    # frest -> f_rest_ のマッピング
    frest_indices = []
    for name in property_names:
        if name.startswith('frest'):
            index = name[5:]  # "frest0" -> "0"
            name_mapping[name] = f'f_rest_{index}'
            frest_indices.append(int(index))
    
    # 残りの属性のマッピング
    if 'opacity' in property_names:
        name_mapping['opacity'] = 'opacity'
    
    # scale -> scale_ のマッピング
    for name in property_names:
        if name.startswith('scale') and len(name) > 5:
            index = name[5:]  # "scale0" -> "0"
            name_mapping[name] = f'scale_{index}'
    
    # rot -> rot_ のマッピング
    for name in property_names:
        if name.startswith('rot') and len(name) > 3:
            index = name[3:]  # "rot0" -> "0"
            name_mapping[name] = f'rot_{index}'
    
    print(f"プロパティマッピング: {name_mapping}")
    
    # データ抽出
    # 位置座標
    positions = np.zeros((num_points, 3))
    for i, coord in enumerate(['x', 'y', 'z']):
        if coord in property_names:
            positions[:, i] = vertex_data[coord]
    
    # 法線情報
    normals = np.zeros((num_points, 3))
    for i, coord in enumerate(['nx', 'ny', 'nz']):
        if coord in property_names:
            normals[:, i] = vertex_data[coord]
    
    # SH係数の抽出
    max_dc_index = max(fdc_indices) if fdc_indices else -1
    f_dc = np.zeros((num_points, max(3, max_dc_index + 1)))  # 最低3つのDC係数が必要
    
    for i in range(f_dc.shape[1]):
        dc_key = f'fdc{i}'
        if dc_key in property_names:
            f_dc[:, i] = vertex_data[dc_key]
    
    # DC係数の確認と調整 - 色情報として使用される最初の3つのDC係数
    # SuperSplatが期待するSH表現に合わせて調整する
    if f_dc.shape[1] >= 3:
        # DC係数の統計を取得
        dc_min = f_dc[:, :3].min(axis=0)
        dc_max = f_dc[:, :3].max(axis=0)
        dc_mean = f_dc[:, :3].mean(axis=0)
        dc_std = f_dc[:, :3].std(axis=0)
        
        print(f"DC係数の統計:")
        print(f"  最小値: {dc_min}")
        print(f"  最大値: {dc_max}")
        print(f"  平均値: {dc_mean}")
        print(f"  標準偏差: {dc_std}")
        
        # 値がゼロまたは非常に小さい場合は、デフォルトの色を追加
        if np.all(np.abs(f_dc[:, :3]) < 0.01):
            print("警告: DC係数がほぼゼロです。デフォルトの色を設定します。")
            # 白色のためのSH係数を設定（SuperSplatの期待する形式に基づく）
            f_dc[:, 0] = 1.0  # 赤
            f_dc[:, 1] = 1.0  # 緑
            f_dc[:, 2] = 1.0  # 青
        
        # 値のスケーリングが必要な場合
        f_dc[:, :3] *= sh_scale
    
    # Rest SH係数
    max_rest_index = max(frest_indices) if frest_indices else -1
    # SuperSplatは45個のrest係数を期待
    num_rest_required = 45
    f_rest = np.zeros((num_points, num_rest_required))
    
    for i in range(min(max_rest_index + 1, num_rest_required)):
        rest_key = f'frest{i}'
        if rest_key in property_names:
            f_rest[:, i] = vertex_data[rest_key]
    
    # 不透明度
    opacities = np.ones(num_points)  # デフォルトは1.0（完全に不透明）
    if 'opacity' in property_names:
        opacities = vertex_data['opacity']
    
    # スケール情報
    scales = np.ones((num_points, 3)) * 0.01  # デフォルト値
    for i in range(3):
        scale_key = f'scale{i}'
        if scale_key in property_names:
            scales[:, i] = vertex_data[scale_key]
    
    # 回転情報（四元数）
    rotations = np.zeros((num_points, 4))
    rotations[:, 0] = 1.0  # デフォルトは単位四元数 [w=1, x=0, y=0, z=0]
    for i in range(4):
        rot_key = f'rot{i}'
        if rot_key in property_names:
            rotations[:, i] = vertex_data[rot_key]
    
    # SuperSplatと互換性のあるPLYファイルを作成
    with open(output_file, 'wb' if binary else 'w') as f:
        # ヘッダー
        if binary:
            f.write(b"ply\n")
            f.write(b"format binary_little_endian 1.0\n")
            f.write(f"element vertex {num_points}\n".encode())
        else:
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write(f"element vertex {num_points}\n")
        
        # SuperSplatの標準プロパティ順序
        properties = [
            "property float x",
            "property float y",
            "property float z",
            "property float nx",
            "property float ny",
            "property float nz",
            "property float f_dc_0",
            "property float f_dc_1",
            "property float f_dc_2"
        ]
        
        # f_rest_0 から f_rest_44 までのプロパティを追加
        for i in range(num_rest_required):
            properties.append(f"property float f_rest_{i}")
        
        # 残りのプロパティ
        properties.extend([
            "property float opacity",
            "property float scale_0",
            "property float scale_1",
            "property float scale_2",
            "property float rot_0",
            "property float rot_1",
            "property float rot_2",
            "property float rot_3"
        ])
        
        # プロパティをファイルに書き込む
        for prop in properties:
            if binary:
                f.write(f"{prop}\n".encode())
            else:
                f.write(f"{prop}\n")
        
        # ヘッダー終了
        if binary:
            f.write(b"end_header\n")
        else:
            f.write("end_header\n")
        
        # データの書き込み
        for i in range(num_points):
            if binary:
                # 位置
                f.write(struct.pack("<fff", float(positions[i, 0]), float(positions[i, 1]), float(positions[i, 2])))
                
                # 法線
                f.write(struct.pack("<fff", float(normals[i, 0]), float(normals[i, 1]), float(normals[i, 2])))
                
                # f_dc係数
                f.write(struct.pack("<fff", float(f_dc[i, 0]), float(f_dc[i, 1]), float(f_dc[i, 2])))
                
                # f_rest係数
                for j in range(num_rest_required):
                    f.write(struct.pack("<f", float(f_rest[i, j])))
                
                # 不透明度
                f.write(struct.pack("<f", float(opacities[i])))
                
                # スケール
                f.write(struct.pack("<fff", float(scales[i, 0]), float(scales[i, 1]), float(scales[i, 2])))
                
                # 回転四元数
                f.write(struct.pack("<ffff", float(rotations[i, 0]), float(rotations[i, 1]), 
                                          float(rotations[i, 2]), float(rotations[i, 3])))
            else:
                # ASCII形式での書き込み
                line_parts = []
                
                # 位置
                line_parts.extend([str(positions[i, 0]), str(positions[i, 1]), str(positions[i, 2])])
                
                # 法線
                line_parts.extend([str(normals[i, 0]), str(normals[i, 1]), str(normals[i, 2])])
                
                # f_dc係数
                line_parts.extend([str(f_dc[i, 0]), str(f_dc[i, 1]), str(f_dc[i, 2])])
                
                # f_rest係数
                for j in range(num_rest_required):
                    line_parts.append(str(f_rest[i, j]))
                
                # 不透明度
                line_parts.append(str(opacities[i]))
                
                # スケール
                line_parts.extend([str(scales[i, 0]), str(scales[i, 1]), str(scales[i, 2])])
                
                # 回転四元数
                line_parts.extend([str(rotations[i, 0]), str(rotations[i, 1]), 
                                  str(rotations[i, 2]), str(rotations[i, 3])])
                
                # 1行としてまとめて書き込み
                f.write(" ".join(line_parts) + "\n")
    
    print(f"変換が完了しました。出力ファイル: {output_file}")

    
input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_custom_to_supersplat(input_file, output_file) # デフォルトはバイナリ出力

入力ファイル: point_cloud_yuya.ply
出力ファイル: point_cloud_yuya_converted.ply
出力形式: バイナリ
SH係数スケール: 1.0
頂点数: 2000000
利用可能な属性: x, y, z, nx, ny, nz, f_dc_0, f_dc_1, f_dc_2, f_rest_0...
プロパティマッピング: {'x': 'x', 'y': 'y', 'z': 'z', 'nx': 'nx', 'ny': 'ny', 'nz': 'nz', 'opacity': 'opacity', 'scale_0': 'scale__0', 'scale_1': 'scale__1', 'scale_2': 'scale__2', 'rot_0': 'rot__0', 'rot_1': 'rot__1', 'rot_2': 'rot__2', 'rot_3': 'rot__3'}
DC係数の統計:
  最小値: [0. 0. 0.]
  最大値: [0. 0. 0.]
  平均値: [0. 0. 0.]
  標準偏差: [0. 0. 0.]
警告: DC係数がほぼゼロです。デフォルトの色を設定します。
変換が完了しました。出力ファイル: point_cloud_yuya_converted.ply


In [1]:
###########################################
# Convert file
###########################################
import pandas as pd
import numpy as np
from plyfile import PlyData, PlyElement

path = "point_cloud_yuya.ply"

# Columns in parquet file
# 'x', 'y', 'z',
# 'cov_q0', 'cov_q1', 'cov_q2', 'cov_q3',
# 'cov_s0', 'cov_s1', 'cov_s2',
# 'alpha0',
# 'r_sh0', 'r_sh1', 'r_sh2', 'r_sh3', 'r_sh4', 'r_sh5', 'r_sh6', 'r_sh7', 'r_sh8', 'r_sh9', 'r_sh10', 'r_sh11', 'r_sh12', 'r_sh13', 'r_sh14', 'r_sh15',
# 'g_sh0', 'g_sh1', 'g_sh2', 'g_sh3', 'g_sh4', 'g_sh5', 'g_sh6', 'g_sh7', 'g_sh8', 'g_sh9', 'g_sh10', 'g_sh11', 'g_sh12', 'g_sh13', 'g_sh14', 'g_sh15',
# 'b_sh0', 'b_sh1', 'b_sh2', 'b_sh3', 'b_sh4', 'b_sh5', 'b_sh6', 'b_sh7', 'b_sh8', 'b_sh9', 'b_sh10', 'b_sh11', 'b_sh12', 'b_sh13', 'b_sh14', 'b_sh15'
df = pd.read_parquet(path)

# Columns in ply file
columns_converted = [
    'x', 'y', 'z',
    'nx','ny','nz',
    'r_sh0', 'g_sh0', 'b_sh0',
    'r_sh1', 'r_sh2', 'r_sh3', 'r_sh4', 'r_sh5', 'r_sh6', 'r_sh7', 'r_sh8', 'r_sh9', 'r_sh10', 'r_sh11', 'r_sh12', 'r_sh13', 'r_sh14', 'r_sh15',
    'g_sh1', 'g_sh2', 'g_sh3', 'g_sh4', 'g_sh5', 'g_sh6', 'g_sh7', 'g_sh8', 'g_sh9', 'g_sh10', 'g_sh11', 'g_sh12', 'g_sh13', 'g_sh14', 'g_sh15',
    'b_sh1', 'b_sh2', 'b_sh3', 'b_sh4', 'b_sh5', 'b_sh6', 'b_sh7', 'b_sh8', 'b_sh9', 'b_sh10', 'b_sh11', 'b_sh12', 'b_sh13', 'b_sh14', 'b_sh15',
    'alpha0',
    'cov_s0', 'cov_s1', 'cov_s2',
    'cov_q3', 'cov_q0', 'cov_q1', 'cov_q2'
]
df = df.reindex(columns=columns_converted, fill_value=0)

# save as ply
PlyData([PlyElement.describe(np.array(list(map(tuple, df.to_numpy())), dtype=[(label, np.float32) for label in columns_converted]), 'vertex')]).write(path + ".ply")

###########################################
# Download file
###########################################
files.download(path + ".ply")

###########################################
# Third Party License Notice
###########################################
# This conversion script was written using the following open source software:
#
# taichi_3d_gaussian_splatting
# https://github.com/wanmeihuali/taichi_3d_gaussian_splatting
#
#                                 Apache License
#                           Version 2.0, January 2004
#                        http://www.apache.org/licenses/
#
#   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION
#
#   1. Definitions.
#
#      "License" shall mean the terms and conditions for use, reproduction,
#      and distribution as defined by Sections 1 through 9 of this document.
#
#      "Licensor" shall mean the copyright owner or entity authorized by
#      the copyright owner that is granting the License.
#
#      "Legal Entity" shall mean the union of the acting entity and all
#      other entities that control, are controlled by, or are under common
#      control with that entity. For the purposes of this definition,
#      "control" means (i) the power, direct or indirect, to cause the
#      direction or management of such entity, whether by contract or
#      otherwise, or (ii) ownership of fifty percent (50%) or more of the
#      outstanding shares, or (iii) beneficial ownership of such entity.
#
#      "You" (or "Your") shall mean an individual or Legal Entity
#      exercising permissions granted by this License.
#
#      "Source" form shall mean the preferred form for making modifications,
#      including but not limited to software source code, documentation
#      source, and configuration files.
#
#      "Object" form shall mean any form resulting from mechanical
#      transformation or translation of a Source form, including but
#      not limited to compiled object code, generated documentation,
#      and conversions to other media types.
#
#      "Work" shall mean the work of authorship, whether in Source or
#      Object form, made available under the License, as indicated by a
#      copyright notice that is included in or attached to the work
#      (an example is provided in the Appendix below).
#
#      "Derivative Works" shall mean any work, whether in Source or Object
#      form, that is based on (or derived from) the Work and for which the
#      editorial revisions, annotations, elaborations, or other modifications
#      represent, as a whole, an original work of authorship. For the purposes
#      of this License, Derivative Works shall not include works that remain
#      separable from, or merely link (or bind by name) to the interfaces of,
#      the Work and Derivative Works thereof.
#
#      "Contribution" shall mean any work of authorship, including
#      the original version of the Work and any modifications or additions
#      to that Work or Derivative Works thereof, that is intentionally
#      submitted to Licensor for inclusion in the Work by the copyright owner
#      or by an individual or Legal Entity authorized to submit on behalf of
#      the copyright owner. For the purposes of this definition, "submitted"
#      means any form of electronic, verbal, or written communication sent
#      to the Licensor or its representatives, including but not limited to
#      communication on electronic mailing lists, source code control systems,
#      and issue tracking systems that are managed by, or on behalf of, the
#      Licensor for the purpose of discussing and improving the Work, but
#      excluding communication that is conspicuously marked or otherwise
#      designated in writing by the copyright owner as "Not a Contribution."
#
#      "Contributor" shall mean Licensor and any individual or Legal Entity
#      on behalf of whom a Contribution has been received by Licensor and
#      subsequently incorporated within the Work.
#
#   2. Grant of Copyright License. Subject to the terms and conditions of
#      this License, each Contributor hereby grants to You a perpetual,
#      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
#      copyright license to reproduce, prepare Derivative Works of,
#      publicly display, publicly perform, sublicense, and distribute the
#      Work and such Derivative Works in Source or Object form.
#
#   3. Grant of Patent License. Subject to the terms and conditions of
#      this License, each Contributor hereby grants to You a perpetual,
#      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
#      (except as stated in this section) patent license to make, have made,
#      use, offer to sell, sell, import, and otherwise transfer the Work,
#      where such license applies only to those patent claims licensable
#      by such Contributor that are necessarily infringed by their
#      Contribution(s) alone or by combination of their Contribution(s)
#      with the Work to which such Contribution(s) was submitted. If You
#      institute patent litigation against any entity (including a
#      cross-claim or counterclaim in a lawsuit) alleging that the Work
#      or a Contribution incorporated within the Work constitutes direct
#      or contributory patent infringement, then any patent licenses
#      granted to You under this License for that Work shall terminate
#      as of the date such litigation is filed.
#
#   4. Redistribution. You may reproduce and distribute copies of the
#      Work or Derivative Works thereof in any medium, with or without
#      modifications, and in Source or Object form, provided that You
#      meet the following conditions:
#
#      (a) You must give any other recipients of the Work or
#          Derivative Works a copy of this License; and
#
#      (b) You must cause any modified files to carry prominent notices
#          stating that You changed the files; and
#
#      (c) You must retain, in the Source form of any Derivative Works
#          that You distribute, all copyright, patent, trademark, and
#          attribution notices from the Source form of the Work,
#          excluding those notices that do not pertain to any part of
#          the Derivative Works; and
#
#      (d) If the Work includes a "NOTICE" text file as part of its
#          distribution, then any Derivative Works that You distribute must
#          include a readable copy of the attribution notices contained
#          within such NOTICE file, excluding those notices that do not
#          pertain to any part of the Derivative Works, in at least one
#          of the following places: within a NOTICE text file distributed
#          as part of the Derivative Works; within the Source form or
#          documentation, if provided along with the Derivative Works; or,
#          within a display generated by the Derivative Works, if and
#          wherever such third-party notices normally appear. The contents
#          of the NOTICE file are for informational purposes only and
#          do not modify the License. You may add Your own attribution
#          notices within Derivative Works that You distribute, alongside
#          or as an addendum to the NOTICE text from the Work, provided
#          that such additional attribution notices cannot be construed
#          as modifying the License.
#
#      You may add Your own copyright statement to Your modifications and
#      may provide additional or different license terms and conditions
#      for use, reproduction, or distribution of Your modifications, or
#      for any such Derivative Works as a whole, provided Your use,
#      reproduction, and distribution of the Work otherwise complies with
#      the conditions stated in this License.
#
#   5. Submission of Contributions. Unless You explicitly state otherwise,
#      any Contribution intentionally submitted for inclusion in the Work
#      by You to the Licensor shall be under the terms and conditions of
#      this License, without any additional terms or conditions.
#      Notwithstanding the above, nothing herein shall supersede or modify
#      the terms of any separate license agreement you may have executed
#      with Licensor regarding such Contributions.
#
#   6. Trademarks. This License does not grant permission to use the trade
#      names, trademarks, service marks, or product names of the Licensor,
#      except as required for reasonable and customary use in describing the
#      origin of the Work and reproducing the content of the NOTICE file.
#
#   7. Disclaimer of Warranty. Unless required by applicable law or
#      agreed to in writing, Licensor provides the Work (and each
#      Contributor provides its Contributions) on an "AS IS" BASIS,
#      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
#      implied, including, without limitation, any warranties or conditions
#      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
#      PARTICULAR PURPOSE. You are solely responsible for determining the
#      appropriateness of using or redistributing the Work and assume any
#      risks associated with Your exercise of permissions under this License.
#
#   8. Limitation of Liability. In no event and under no legal theory,
#      whether in tort (including negligence), contract, or otherwise,
#      unless required by applicable law (such as deliberate and grossly
#      negligent acts) or agreed to in writing, shall any Contributor be
#      liable to You for damages, including any direct, indirect, special,
#      incidental, or consequential damages of any character arising as a
#      result of this License or out of the use or inability to use the
#      Work (including but not limited to damages for loss of goodwill,
#      work stoppage, computer failure or malfunction, or any and all
#      other commercial damages or losses), even if such Contributor
#      has been advised of the possibility of such damages.
#
#   9. Accepting Warranty or Additional Liability. While redistributing
#      the Work or Derivative Works thereof, You may choose to offer,
#      and charge a fee for, acceptance of support, warranty, indemnity,
#      or other liability obligations and/or rights consistent with this
#      License. However, in accepting such obligations, You may act only
#      on Your own behalf and on Your sole responsibility, not on behalf
#      of any other Contributor, and only if You agree to indemnify,
#      defend, and hold each Contributor harmless for any liability
#      incurred by, or claims asserted against, such Contributor by reason
#      of your accepting any such warranty or additional liability.
#
#   END OF TERMS AND CONDITIONS
#
#   APPENDIX: How to apply the Apache License to your work.
#
#      To apply the Apache License to your work, attach the following
#      boilerplate notice, with the fields enclosed by brackets "[]"
#      replaced with your own identifying information. (Don't include
#      the brackets!)  The text should be enclosed in the appropriate
#      comment syntax for the file format. We also recommend that a
#      file or class name and description of purpose be included on the
#      same "printed page" as the copyright notice for easier
#      identification within third-party archives.
#
#   Copyright [yyyy] [name of copyright owner]
#
#   Licensed under the Apache License, Version 2.0 (the "License");
#   you may not use this file except in compliance with the License.
#   You may obtain a copy of the License at
#
#       http://www.apache.org/licenses/LICENSE-2.0
#
#   Unless required by applicable law or agreed to in writing, software
#   distributed under the License is distributed on an "AS IS" BASIS,
#   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#   See the License for the specific language governing permissions and
#   limitations under the License.

###########################################
# License
###########################################
# This conversion script is licensed under the Apache License 2.0 License.
#
# Copyright 2023 Akiya Research Institute, Inc.
#
# Licensed under the Apache License, Version 2.0 (the “License”);
# you may not use this script except in compliance with the License.
# You may obtain a copy of the License at
# http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an “AS IS” BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#
# See the License for the specific language governing permissions and
# limitations under the License.

ArrowInvalid: Could not open Parquet input source '<Buffer>': Parquet magic bytes not found in footer. Either the file is corrupted or this is not a parquet file.

In [12]:
import numpy as np
import open3d as o3d
import argparse
import os
import struct
from plyfile import PlyData


def convert_to_exact_supersplat_format(input_file, output_file, binary=True):
    """
    入力PLYファイルを正確なSuperSplat形式に変換する
    
    Args:
        input_file (str): 入力PLYファイルへのパス
        output_file (str): 出力SuperSplat互換PLYファイルへのパス
        binary (bool): バイナリ形式で出力するかどうか（デフォルトはバイナリ）
    """
    print(f"入力ファイル: {input_file}")
    print(f"出力ファイル: {output_file}")
    print(f"出力形式: {'バイナリ' if binary else 'ASCII'}")
    
    # PLYファイルを読み込む
    ply_data = PlyData.read(input_file)
    vertex_data = ply_data['vertex']
    
    # データ数を取得
    num_points = len(vertex_data)
    print(f"頂点数: {num_points}")
    
    # 利用可能な属性名のリストを取得
    property_names = [prop.name for prop in vertex_data.properties]
    print(f"利用可能な属性: {', '.join(property_names[:10])}...")
    
    # 基本的な位置情報を取得
    x = np.array(vertex_data['x'])
    y = np.array(vertex_data['y'])
    z = np.array(vertex_data['z'])
    positions = np.column_stack((x, y, z))
    
    # 法線情報を取得（存在する場合）
    try:
        nx = np.array(vertex_data['nx'])
        ny = np.array(vertex_data['ny'])
        nz = np.array(vertex_data['nz'])
        normals = np.column_stack((nx, ny, nz))
    except ValueError:
        print("警告: 法線情報が見つかりません。ゼロで初期化します。")
        normals = np.zeros_like(positions)
    
    # SH係数を抽出
    # DC係数 (最初の3つの球面調和関数係数)
    f_dc = np.zeros((num_points, 3))
    for i in range(3):
        dc_key = f'f_dc_{i}'
        if dc_key in property_names:
            f_dc[:, i] = np.array(vertex_data[dc_key])
        else:
            print(f"警告: {dc_key}が見つかりません。値を初期化します。")
            # 色に対応するDC係数のデフォルト値を設定
            # シグモイド(0)=0.5なので、これは中程度のグレーに相当
            f_dc[:, i] = 0.0
    
    # Rest係数 (残りの球面調和関数係数)
    f_rest = np.zeros((num_points, 45))  # Maxで45個のrest係数
    for i in range(45):
        rest_key = f'f_rest_{i}'
        if rest_key in property_names:
            f_rest[:, i] = np.array(vertex_data[rest_key])
        else:
            # 見つからない係数は0で初期化
            if i < 5:  # 基本的なものだけ警告表示
                print(f"警告: {rest_key}が見つかりません。0で初期化します。")
    
    # 不透明度を取得
    try:
        opacities = np.array(vertex_data['opacity'])
    except ValueError:
        print("警告: 不透明度情報が見つかりません。デフォルト値で初期化します。")
        opacities = np.ones(num_points)
    
    # スケール情報を取得
    scales = np.zeros((num_points, 3))
    try:
        for i in range(3):
            scale_key = f'scale_{i}'
            if scale_key in property_names:
                scales[:, i] = np.array(vertex_data[scale_key])
            else:
                print(f"警告: {scale_key}が見つかりません。デフォルト値で初期化します。")
                scales[:, i] = 0.01  # デフォルト値
    except Exception as e:
        print(f"スケール情報の抽出中にエラーが発生しました: {e}")
        scales = np.ones((num_points, 3)) * 0.01
    
    # 回転情報（四元数）を取得
    rotations = np.zeros((num_points, 4))
    try:
        for i in range(4):
            rot_key = f'rot_{i}'
            if rot_key in property_names:
                rotations[:, i] = np.array(vertex_data[rot_key])
            else:
                print(f"警告: {rot_key}が見つかりません。デフォルト値で初期化します。")
                if i == 0:
                    rotations[:, i] = 1.0  # 単位四元数のw成分
    except Exception as e:
        print(f"回転情報の抽出中にエラーが発生しました: {e}")
        rotations[:, 0] = 1.0  # 単位四元数（w, x, y, z）= (1, 0, 0, 0)
    
    # SuperSplat形式のPLYファイルを作成
    with open(output_file, 'wb' if binary else 'w') as f:
        # ヘッダー部分
        if binary:
            f.write(b"ply\n")
            f.write(b"format binary_little_endian 1.0\n")
            f.write(f"element vertex {num_points}\n".encode())
        else:
            f.write("ply\n")
            f.write("format ascii 1.0\n")
            f.write(f"element vertex {num_points}\n")
        
        # SuperSplatのヘッダー情報に完全に一致するプロパティリスト
        properties = [
            "property float x",
            "property float y",
            "property float z",
            "property float nx",
            "property float ny",
            "property float nz",
            "property float f_dc_0",
            "property float f_dc_1",
            "property float f_dc_2",
        ]
        
        # f_rest_0 から f_rest_44 までのプロパティを追加
        for i in range(45):
            properties.append(f"property float f_rest_{i}")
        
        # 残りのプロパティ
        properties.extend([
            "property float opacity",
            "property float scale_0",
            "property float scale_1",
            "property float scale_2",
            "property float rot_0",
            "property float rot_1",
            "property float rot_2",
            "property float rot_3",
        ])
        
        # プロパティをファイルに書き込む
        for prop in properties:
            if binary:
                f.write(f"{prop}\n".encode())
            else:
                f.write(f"{prop}\n")
        
        # ヘッダー終了
        if binary:
            f.write(b"end_header\n")
        else:
            f.write("end_header\n")
        
        # データ部分
        for i in range(num_points):
            if binary:
                # 位置
                f.write(struct.pack("<fff", float(positions[i, 0]), float(positions[i, 1]), float(positions[i, 2])))
                
                # 法線
                f.write(struct.pack("<fff", float(normals[i, 0]), float(normals[i, 1]), float(normals[i, 2])))
                
                # f_dc_0, f_dc_1, f_dc_2
                f.write(struct.pack("<fff", float(f_dc[i, 0]), float(f_dc[i, 1]), float(f_dc[i, 2])))
                
                # f_rest_0 から f_rest_44 まで
                for j in range(45):
                    value = float(f_rest[i, j]) if j < f_rest.shape[1] else 0.0
                    f.write(struct.pack("<f", value))
                
                # 不透明度
                f.write(struct.pack("<f", float(opacities[i])))
                
                # スケール
                f.write(struct.pack("<fff", float(scales[i, 0]), float(scales[i, 1]), float(scales[i, 2])))
                
                # 回転四元数
                f.write(struct.pack("<ffff", float(rotations[i, 0]), float(rotations[i, 1]), 
                                         float(rotations[i, 2]), float(rotations[i, 3])))
            else:
                # ASCII形式での書き込み
                line_parts = []
                
                # 位置
                line_parts.extend([str(positions[i, 0]), str(positions[i, 1]), str(positions[i, 2])])
                
                # 法線
                line_parts.extend([str(normals[i, 0]), str(normals[i, 1]), str(normals[i, 2])])
                
                # f_dc_0, f_dc_1, f_dc_2
                line_parts.extend([str(f_dc[i, 0]), str(f_dc[i, 1]), str(f_dc[i, 2])])
                
                # f_rest_0 から f_rest_44 まで
                for j in range(45):
                    value = f_rest[i, j] if j < f_rest.shape[1] else 0.0
                    line_parts.append(str(value))
                
                # 不透明度
                line_parts.append(str(opacities[i]))
                
                # スケール
                line_parts.extend([str(scales[i, 0]), str(scales[i, 1]), str(scales[i, 2])])
                
                # 回転四元数
                line_parts.extend([str(rotations[i, 0]), str(rotations[i, 1]), 
                                  str(rotations[i, 2]), str(rotations[i, 3])])
                
                # 1行としてまとめて書き込み
                f.write(" ".join(line_parts) + "\n")
    
    print(f"変換が完了しました。出力ファイル: {output_file}")

input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_to_exact_supersplat_format(input_file, output_file) 

入力ファイル: point_cloud_yuya.ply
出力ファイル: point_cloud_yuya_converted.ply
出力形式: バイナリ
頂点数: 2000000
利用可能な属性: x, y, z, nx, ny, nz, f_dc_0, f_dc_1, f_dc_2, f_rest_0...
変換が完了しました。出力ファイル: point_cloud_yuya_converted.ply


In [8]:
import numpy as np
import open3d as o3d
import argparse
import os

def convert_custom_to_3dgs(input_file, output_file):
    """
    カスタム実装から出力されたPLYファイルをSuperSplatと互換性のある3DGS形式に変換する
    
    Args:
        input_file (str): 入力カスタムPLYファイルへのパス
        output_file (str): 出力3DGS PLYファイルへのパス
    """
    print(f"入力ファイル: {input_file}")
    print(f"出力ファイル: {output_file}")
    
    # PLYファイルを読み込む
    ply_data = PlyData.read(input_file)
    vertex_data = ply_data['vertex']
    
    # データ数を取得
    num_points = len(vertex_data)
    print(f"頂点数: {num_points}")
    
    # 利用可能な属性名のリストを取得
    property_names = [prop.name for prop in vertex_data.properties]
    print(f"利用可能な属性: {property_names}")
    
    # 基本的な位置情報を取得
    x = np.array(vertex_data['x'])
    y = np.array(vertex_data['y'])
    z = np.array(vertex_data['z'])
    positions = np.column_stack((x, y, z))
    
    # 法線情報を取得（存在する場合）
    try:
        nx = np.array(vertex_data['nx'])
        ny = np.array(vertex_data['ny'])
        nz = np.array(vertex_data['nz'])
        normals = np.column_stack((nx, ny, nz))
    except ValueError:
        print("警告: 法線情報が見つかりません。ゼロで初期化します。")
        normals = np.zeros_like(positions)
    
    # 不透明度を取得
    try:
        opacities = np.array(vertex_data['opacity'])
    except ValueError:
        print("警告: 不透明度情報が見つかりません。デフォルト値で初期化します。")
        opacities = np.ones(num_points)
    
    # スケール情報を取得
    scales = np.zeros((num_points, 3))
    try:
        for i in range(3):  # 3D Gaussianなので3軸のスケール
            scale_key = f'scale_{i}'
            if scale_key in property_names:
                scales[:, i] = np.array(vertex_data[scale_key])
            else:
                print(f"警告: {scale_key}が見つかりません。デフォルト値で初期化します。")
                scales[:, i] = 0.01  # デフォルト値
    except Exception as e:
        print(f"スケール情報の抽出中にエラーが発生しました: {e}")
        scales = np.ones((num_points, 3)) * 0.01
    
    # 回転情報（四元数）を取得
    rotations = np.zeros((num_points, 4))
    try:
        for i in range(4):  # 四元数は4成分
            rot_key = f'rot_{i}'
            if rot_key in property_names:
                rotations[:, i] = np.array(vertex_data[rot_key])
            else:
                print(f"警告: {rot_key}が見つかりません。デフォルト値で初期化します。")
                if i == 0:
                    rotations[:, i] = 1.0  # 単位四元数のw成分
    except Exception as e:
        print(f"回転情報の抽出中にエラーが発生しました: {e}")
        rotations[:, 0] = 1.0  # 単位四元数（w, x, y, z）= (1, 0, 0, 0)
    
    # SH係数を抽出
    sh_dc_indices = [i for i in range(len(property_names)) if property_names[i].startswith('f_dc_')]
    sh_rest_indices = [i for i in range(len(property_names)) if property_names[i].startswith('f_rest_')]
    
    num_sh_dc = len(sh_dc_indices)
    num_sh_rest = len(sh_rest_indices)
    
    print(f"DC SH係数数: {num_sh_dc}")
    print(f"残りのSH係数数: {num_sh_rest}")
    
    # SH係数からRGB色を計算（最初の3つのDC係数がRGBに対応）
    sh_dc = np.zeros((num_points, num_sh_dc))
    for i, idx in enumerate(sh_dc_indices):
        sh_dc[:, i] = np.array(vertex_data[property_names[idx]])
    
    # DC係数の最初の3つ（各色チャンネルに1つずつ）から色を抽出
    # 3DGSでは通常、SHの0次係数の最初の3つがRGBに対応
    if num_sh_dc >= 3:
        # シグモイド関数を適用して0-1の範囲に収める
        colors = np.zeros((num_points, 3))
        for i in range(min(3, num_sh_dc)):
            colors[:, i] = 1.0 / (1.0 + np.exp(-sh_dc[:, i]))
    else:
        print("警告: DC SH係数が3未満です。白色で初期化します。")
        colors = np.ones((num_points, 3))
    
    # ---- SuperSplat形式のplyデータ構造を作成 ----
    
    # 必要なデータを構造化
    dtype_list = [
        ('x', 'f4'), ('y', 'f4'), ('z', 'f4'),           # 位置
        ('nx', 'f4'), ('ny', 'f4'), ('nz', 'f4'),        # 法線
        ('red', 'u1'), ('green', 'u1'), ('blue', 'u1'),  # 色
        ('scale_0', 'f4'), ('scale_1', 'f4'), ('scale_2', 'f4'),  # スケール
        ('rot_0', 'f4'), ('rot_1', 'f4'), ('rot_2', 'f4'), ('rot_3', 'f4'),  # 四元数
        ('opacity', 'f4')                                 # 不透明度
    ]
    
    # SH係数の型情報を追加
    for i in range(num_sh_dc):
        dtype_list.append((f'f_dc_{i}', 'f4'))
    
    for i in range(num_sh_rest):
        dtype_list.append((f'f_rest_{i}', 'f4'))
    
    # 構造化配列を作成
    vertex_all = np.zeros(num_points, dtype=dtype_list)
    
    # データを設定
    vertex_all['x'] = positions[:, 0]
    vertex_all['y'] = positions[:, 1]
    vertex_all['z'] = positions[:, 2]
    
    vertex_all['nx'] = normals[:, 0]
    vertex_all['ny'] = normals[:, 1]
    vertex_all['nz'] = normals[:, 2]
    
    # 色を0-255の範囲に変換
    vertex_all['red'] = np.clip(colors[:, 0] * 255, 0, 255).astype(np.uint8)
    vertex_all['green'] = np.clip(colors[:, 1] * 255, 0, 255).astype(np.uint8)
    vertex_all['blue'] = np.clip(colors[:, 2] * 255, 0, 255).astype(np.uint8)
    
    vertex_all['scale_0'] = scales[:, 0]
    vertex_all['scale_1'] = scales[:, 1]
    vertex_all['scale_2'] = scales[:, 2]
    
    vertex_all['rot_0'] = rotations[:, 0]
    vertex_all['rot_1'] = rotations[:, 1]
    vertex_all['rot_2'] = rotations[:, 2]
    vertex_all['rot_3'] = rotations[:, 3]
    
    vertex_all['opacity'] = opacities
    
    # SH係数を設定
    for i, idx in enumerate(sh_dc_indices):
        dc_key = property_names[idx]
        vertex_all[f'f_dc_{i}'] = np.array(vertex_data[dc_key])
    
    for i, idx in enumerate(sh_rest_indices):
        rest_key = property_names[idx]
        vertex_all[f'f_rest_{i}'] = np.array(vertex_data[rest_key])
    
    # PLYElement作成
    vertex_element = PlyElement.describe(vertex_all, 'vertex')
    
    # PLYファイル書き出し
    ply_data_out = PlyData([vertex_element], text=True)
    ply_data_out.write(output_file)
    
    print(f"変換が完了しました。出力ファイル: {output_file}")
    
    
input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_custom_to_3dgs(input_file, output_file) 

入力ファイル: point_cloud_yuya.ply
出力ファイル: point_cloud_yuya_converted.ply
頂点数: 2000000
利用可能な属性: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
DC SH係数数: 3
残りのSH係数数: 45
変換が完了しました。出力ファイル: point_cloud_yuya_converted.ply


In [5]:
import numpy as np
import open3d as o3d
import argparse
import os

def convert_custom_to_3dgs(input_file, output_file):
    """
    カスタム実装から出力されたPLYファイルをオリジナルの3DGS形式に変換する
    
    Args:
        input_file (str): 入力カスタムPLYファイルへのパス
        output_file (str): 出力3DGS PLYファイルへのパス
    """
    print(f"入力ファイル: {input_file}")
    print(f"出力ファイル: {output_file}")
    
    # PLYファイルを読み込む
    ply_data = PlyData.read(input_file)
    vertex_data = ply_data['vertex']
    
    # データ数を取得
    num_points = len(vertex_data)
    print(f"頂点数: {num_points}")
    
    # 利用可能な属性名のリストを取得
    property_names = [prop.name for prop in vertex_data.properties]
    print(f"利用可能な属性: {property_names}")
    
    # 基本的な位置情報を取得
    x = np.array(vertex_data['x'])
    y = np.array(vertex_data['y'])
    z = np.array(vertex_data['z'])
    positions = np.column_stack((x, y, z))
    
    # 法線情報を取得（存在する場合）
    try:
        nx = np.array(vertex_data['nx'])
        ny = np.array(vertex_data['ny'])
        nz = np.array(vertex_data['nz'])
        normals = np.column_stack((nx, ny, nz))
    except ValueError:
        print("警告: 法線情報が見つかりません。ゼロで初期化します。")
        normals = np.zeros_like(positions)
    
    # 不透明度を取得
    try:
        opacities = np.array(vertex_data['opacity'])
    except ValueError:
        print("警告: 不透明度情報が見つかりません。デフォルト値で初期化します。")
        opacities = np.ones(num_points)
    
    # スケール情報を取得
    scales = np.zeros((num_points, 3))
    try:
        for i in range(3):  # 3D Gaussianなので3軸のスケール
            scale_key = f'scale_{i}'
            if scale_key in property_names:
                scales[:, i] = np.array(vertex_data[scale_key])
            else:
                print(f"警告: {scale_key}が見つかりません。デフォルト値で初期化します。")
                scales[:, i] = 0.01  # デフォルト値
    except Exception as e:
        print(f"スケール情報の抽出中にエラーが発生しました: {e}")
        scales = np.ones((num_points, 3)) * 0.01
    
    # 回転情報（四元数）を取得
    rotations = np.zeros((num_points, 4))
    try:
        for i in range(4):  # 四元数は4成分
            rot_key = f'rot_{i}'
            if rot_key in property_names:
                rotations[:, i] = np.array(vertex_data[rot_key])
            else:
                print(f"警告: {rot_key}が見つかりません。デフォルト値で初期化します。")
                if i == 0:
                    rotations[:, i] = 1.0  # 単位四元数のw成分
    except Exception as e:
        print(f"回転情報の抽出中にエラーが発生しました: {e}")
        rotations[:, 0] = 1.0  # 単位四元数（w, x, y, z）= (1, 0, 0, 0)
    
    # SH係数を抽出
    sh_dc_indices = [i for i in range(len(property_names)) if property_names[i].startswith('f_dc_')]
    sh_rest_indices = [i for i in range(len(property_names)) if property_names[i].startswith('f_rest_')]
    
    num_sh_dc = len(sh_dc_indices)
    num_sh_rest = len(sh_rest_indices)
    
    print(f"DC SH係数数: {num_sh_dc}")
    print(f"残りのSH係数数: {num_sh_rest}")
    
    # SH係数からRGB色を計算（最初の3つのDC係数がRGBに対応）
    sh_dc = np.zeros((num_points, num_sh_dc))
    for i, idx in enumerate(sh_dc_indices):
        sh_dc[:, i] = np.array(vertex_data[property_names[idx]])
    
    # DC係数の最初の3つ（各色チャンネルに1つずつ）から色を抽出
    # 3DGSでは通常、SHの0次係数の最初の3つがRGBに対応
    if num_sh_dc >= 3:
        # シグモイド関数を適用して0-1の範囲に収める
        colors = np.zeros((num_points, 3))
        for i in range(min(3, num_sh_dc)):
            colors[:, i] = 1.0 / (1.0 + np.exp(-sh_dc[:, i]))
    else:
        print("警告: DC SH係数が3未満です。白色で初期化します。")
        colors = np.ones((num_points, 3))
    
    # 新しいPLYファイルを作成（3DGS形式）
    with open(output_file, 'w') as f:
        # ヘッダー
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {num_points}\n")
        
        # 位置
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        
        # 法線
        f.write("property float nx\n")
        f.write("property float ny\n")
        f.write("property float nz\n")
        
        # 色情報
        f.write("property uchar red\n")
        f.write("property uchar green\n")
        f.write("property uchar blue\n")
        
        # スケール（3DGSではスケールとして保存）
        f.write("property float scale_0\n")
        f.write("property float scale_1\n")
        f.write("property float scale_2\n")
        
        # 回転（四元数）
        f.write("property float rot_0\n")
        f.write("property float rot_1\n")
        f.write("property float rot_2\n")
        f.write("property float rot_3\n")
        
        # 不透明度
        f.write("property float opacity\n")
        
        # DC SH係数（3DGSの特徴量）
        for i in range(num_sh_dc):
            f.write(f"property float f_dc_{i}\n")
        
        # 残りのSH係数（3DGSの特徴量）
        for i in range(num_sh_rest):
            f.write(f"property float f_rest_{i}\n")
        
        # ヘッダー終了
        f.write("end_header\n")
        
        # データの書き込み
        for i in range(num_points):
            # 位置
            f.write(f"{positions[i, 0]} {positions[i, 1]} {positions[i, 2]} ")
            
            # 法線
            f.write(f"{normals[i, 0]} {normals[i, 1]} {normals[i, 2]} ")
            
            # 色情報（0-255の範囲に変換）
            r = int(min(max(colors[i, 0] * 255, 0), 255))
            g = int(min(max(colors[i, 1] * 255, 0), 255))
            b = int(min(max(colors[i, 2] * 255, 0), 255))
            f.write(f"{r} {g} {b} ")
            
            # スケール
            f.write(f"{scales[i, 0]} {scales[i, 1]} {scales[i, 2]} ")
            
            # 回転（四元数）
            f.write(f"{rotations[i, 0]} {rotations[i, 1]} {rotations[i, 2]} {rotations[i, 3]} ")
            
            # 不透明度
            f.write(f"{opacities[i]} ")
            
            # DC SH係数
            for j in range(num_sh_dc):
                sh_dc_key = property_names[sh_dc_indices[j]]
                f.write(f"{vertex_data[sh_dc_key][i]} ")
            
            # 残りのSH係数
            for j in range(num_sh_rest):
                sh_rest_key = property_names[sh_rest_indices[j]]
                f.write(f"{vertex_data[sh_rest_key][i]}")
                
                # 最後の要素の後には改行を入れる
                if j == num_sh_rest - 1:
                    f.write("\n")
                else:
                    f.write(" ")
    
    print(f"変換が完了しました。出力ファイル: {output_file}")


input_file = "point_cloud_yuya.ply"
base_name, ext = os.path.splitext(input_file)
output_file = f"{base_name}_converted{ext}"

convert_custom_to_3dgs(input_file, output_file)    


入力ファイル: point_cloud_yuya.ply
出力ファイル: point_cloud_yuya_converted.ply
頂点数: 2000000
利用可能な属性: ['x', 'y', 'z', 'nx', 'ny', 'nz', 'f_dc_0', 'f_dc_1', 'f_dc_2', 'f_rest_0', 'f_rest_1', 'f_rest_2', 'f_rest_3', 'f_rest_4', 'f_rest_5', 'f_rest_6', 'f_rest_7', 'f_rest_8', 'f_rest_9', 'f_rest_10', 'f_rest_11', 'f_rest_12', 'f_rest_13', 'f_rest_14', 'f_rest_15', 'f_rest_16', 'f_rest_17', 'f_rest_18', 'f_rest_19', 'f_rest_20', 'f_rest_21', 'f_rest_22', 'f_rest_23', 'f_rest_24', 'f_rest_25', 'f_rest_26', 'f_rest_27', 'f_rest_28', 'f_rest_29', 'f_rest_30', 'f_rest_31', 'f_rest_32', 'f_rest_33', 'f_rest_34', 'f_rest_35', 'f_rest_36', 'f_rest_37', 'f_rest_38', 'f_rest_39', 'f_rest_40', 'f_rest_41', 'f_rest_42', 'f_rest_43', 'f_rest_44', 'opacity', 'scale_0', 'scale_1', 'scale_2', 'rot_0', 'rot_1', 'rot_2', 'rot_3']
DC SH係数数: 3
残りのSH係数数: 45
変換が完了しました。出力ファイル: point_cloud_yuya_converted.ply


In [9]:
import pandas as pd
import numpy as np
import open3d as o3d
from plyfile import PlyData, PlyElement

path = "point_cloud_yuya_converted.ply"

# Columns in parquet file
# 'x', 'y', 'z',
# 'cov_q0', 'cov_q1', 'cov_q2', 'cov_q3',
# 'cov_s0', 'cov_s1', 'cov_s2',
# 'alpha0',
# 'r_sh0', 'r_sh1', 'r_sh2', 'r_sh3', 'r_sh4', 'r_sh5', 'r_sh6', 'r_sh7', 'r_sh8', 'r_sh9', 'r_sh10', 'r_sh11', 'r_sh12', 'r_sh13', 'r_sh14', 'r_sh15',
# 'g_sh0', 'g_sh1', 'g_sh2', 'g_sh3', 'g_sh4', 'g_sh5', 'g_sh6', 'g_sh7', 'g_sh8', 'g_sh9', 'g_sh10', 'g_sh11', 'g_sh12', 'g_sh13', 'g_sh14', 'g_sh15',
# 'b_sh0', 'b_sh1', 'b_sh2', 'b_sh3', 'b_sh4', 'b_sh5', 'b_sh6', 'b_sh7', 'b_sh8', 'b_sh9', 'b_sh10', 'b_sh11', 'b_sh12', 'b_sh13', 'b_sh14', 'b_sh15'
#df = pd.read_parquet(path)
pcd = o3d.io.read_point_cloud(path)
o3d.visualization.draw_geometries([pcd])

In [ ]:
# Experimental
def construct_list_of_attributes(self):
    l = ['x', 'y', 'z', 'nx', 'ny', 'nz']
    # All channels except the 3 DC
    for i in range(self.splats["sh0"].shape[1]*self.splats["sh0"].shape[2]):
        l.append('f_dc_{}'.format(i))
    for i in range(self.splats["shN"].shape[1]*self.splats["shN"].shape[2]):
        l.append('f_rest_{}'.format(i))
    l.append('opacity')
    for i in range(self.splats["scales"].shape[1]):
        l.append('scale_{}'.format(i))
    for i in range(self.splats["quats"].shape[1]):
        l.append('rot_{}'.format(i))
    return l

# Experimental
@torch.no_grad()
def save_ply(self, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    xyz = self.splats["means"].detach().cpu().numpy()
    normals = np.zeros_like(xyz)
    f_dc = self.splats["sh0"].detach().transpose(1, 2).flatten(start_dim=1).contiguous().cpu().numpy()
    f_rest = self.splats["shN"].detach().transpose(1, 2).flatten(start_dim=1).contiguous().cpu().numpy()
    opacities = self.splats["opacities"].detach().unsqueeze(-1).cpu().numpy()
    scale = self.splats["scales"].detach().cpu().numpy()
    rotation = self.splats["quats"].detach().cpu().numpy()

    dtype_full = [(attribute, 'f4') for attribute in self.construct_list_of_attributes()]

    elements = np.empty(xyz.shape[0], dtype=dtype_full)
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
    elements[:] = list(map(tuple, attributes))
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(path)

In [ ]:
def save_ply(splats: torch.nn.ParameterDict, dir: str, colors: torch.Tensor = None):
    # Convert all tensors to numpy arrays in one go
    print(f"Saving ply to {dir}")
    numpy_data = {k: v.detach().cpu().numpy() for k, v in splats.items()}

    means = numpy_data["means"]
    scales = numpy_data["scales"]
    quats = numpy_data["quats"]
    opacities = numpy_data["opacities"]

    sh0 = numpy_data["sh0"].transpose(0, 2, 1).reshape(means.shape[0], -1)
    shN = numpy_data["shN"].transpose(0, 2, 1).reshape(means.shape[0], -1)

    # Create a mask to identify rows with NaN or Inf in any of the numpy_data arrays
    invalid_mask = (
        np.isnan(means).any(axis=1)
        | np.isinf(means).any(axis=1)
        | np.isnan(scales).any(axis=1)
        | np.isinf(scales).any(axis=1)
        | np.isnan(quats).any(axis=1)
        | np.isinf(quats).any(axis=1)
        | np.isnan(opacities).any(axis=0)
        | np.isinf(opacities).any(axis=0)
        | np.isnan(sh0).any(axis=1)
        | np.isinf(sh0).any(axis=1)
        | np.isnan(shN).any(axis=1)
        | np.isinf(shN).any(axis=1)
    )

    # Filter out rows with NaNs or Infs from all data arrays
    means = means[~invalid_mask]
    scales = scales[~invalid_mask]
    quats = quats[~invalid_mask]
    opacities = opacities[~invalid_mask]
    sh0 = sh0[~invalid_mask]
    shN = shN[~invalid_mask]

    num_points = means.shape[0]

    with open(dir, "wb") as f:
        # Write PLY header
        f.write(b"ply\n")
        f.write(b"format binary_little_endian 1.0\n")
        f.write(f"element vertex {num_points}\n".encode())
        f.write(b"property float x\n")
        f.write(b"property float y\n")
        f.write(b"property float z\n")
        f.write(b"property float nx\n")
        f.write(b"property float ny\n")
        f.write(b"property float nz\n")

        if colors is not None:
            for j in range(colors.shape[1]):
                f.write(f"property float f_dc_{j}\n".encode())
        else:
            for i, data in enumerate([sh0, shN]):
                prefix = "f_dc" if i == 0 else "f_rest"
                for j in range(data.shape[1]):
                    f.write(f"property float {prefix}_{j}\n".encode())

        f.write(b"property float opacity\n")

        for i in range(scales.shape[1]):
            f.write(f"property float scale_{i}\n".encode())
        for i in range(quats.shape[1]):
            f.write(f"property float rot_{i}\n".encode())

        f.write(b"end_header\n")

        # Write vertex data
        for i in range(num_points):
            f.write(struct.pack("<fff", *means[i]))  # x, y, z
            f.write(struct.pack("<fff", 0, 0, 0))  # nx, ny, nz (zeros)

            if colors is not None:
                color = colors.detach().cpu().numpy()
                for j in range(color.shape[1]):
                    f_dc = (color[i, j] - 0.5) / 0.2820947917738781
                    f.write(struct.pack("<f", f_dc))
            else:
                for data in [sh0, shN]:
                    for j in range(data.shape[1]):
                        f.write(struct.pack("<f", data[i, j]))

            f.write(struct.pack("<f", opacities[i]))  # opacity

            for data in [scales, quats]:
                for j in range(data.shape[1]):
                    f.write(struct.pack("<f", data[i, j]))
